# Module 10 · Differential expression

Pseudobulk, donor-blocked, across the four quadrant contrasts.

**Donor as replicate, not cell.** Cells from one donor are not independent.
Aggregating to one profile per donor per population before testing is what keeps
the p-values interpretable; the alternative inflates the effective n by the
number of cells.

**The four contrasts** derive from `AXIS`, so their names change with it:

```
<X>axis_SnCpos    activation effect within senescent cells
<X>axis_SnCneg    activation effect within non-senescent cells
SnCaxis_<X>pos    senescence effect within activation-high cells
SnCaxis_<X>neg    senescence effect within activation-low cells
```

The first two isolate activation at fixed senescence; the second two isolate
senescence at fixed activation. Comparing them separates the two programmes at
gene level, which is what module 09 does with scores.

| Section | |
|---|---|
| 01-02 | config, gene-set inventory |
| 03 | pseudobulk build — seven steps, each checked |
| 04 | four-contrast limma-voom loop (validated flow) |
| 05-07 | volcanoes, DEG counts, Venns |
| 08-09 | within-axis split, panel enrichment of the hits |
| 10-11 | directional Venns, genome-maintenance genes |
| 12 | axis-depletion check and functional annotation |
| 13 | export rankings for module 11 |

**Design.** `~ pop + grp2 + Sex` with intercept, coefficient `popTEST`,
`duplicateCorrelation` blocking on donor, minimum 10 cells per pseudobulk
sample.

> **Legacy naming.** `DAMaxis_*` appears in contrast and file names — the
> source's tags from when the axis was fixed to DAM. `X_TAG` in the config
> controls it; the literals are left alone so existing results stay findable.

---
## 01 · Config

**Why.** Self-contained — the notebook carries its own config rather than
importing one. Paths come from the environment; see `.env.example`.

**`AXIS` is the one thing you change.** Column names, quadrant labels, contrast
names, figure titles and output directories derive from it, and outputs are
namespaced by axis so runs never overwrite each other.

```
AXIS <- "IRM"    # "IRM" | "DAM_like" | "ARM" | "Stress"
```

`STATE_ORDER` and `STATE_COLORS` name all five states regardless of `AXIS` —
that is the annotation registry, not a per-run choice.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
# Paths come from the environment - see .env.example.
#   SENESCENCE_DATA : analysis root
#   SENESCENCE_REF  : reference root (published panels, read-only)
# =============================================================================
SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")
if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")

# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# CONFIG (Cell §0)
# =============================================================================
# Tests whether SnC cells show cell-cycle arrest and senescence pathway
# enrichment relative to Non-SnC. Reads M04's tissue-wide preprocessed .qs,
# subsets to one CELL_TYPE per run, scores cell cycle (Tirosh) + 10 module
# scores (Sloan Table S9), and runs five complementary models: Wilcoxon,
# RLM, OLS, LMM, and balanced OLS bootstrap.
#
# All run-time decisions live here. All cross-cell helpers are defined here
# so §1–§8 can use them without dependency-ordering issues.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat("§0 — MODULE 05 CONFIG\n")
cat("=", strrep("=", 71), "\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────
options(repr.plot.width = 10, repr.plot.height = 5)


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")
CONDITION_TAG     <- if (IS_AGING) STUDY_TYPE else paste0(STUDY_TYPE, "_", DISEASE)
CONDITION_SUBPATH <- if (IS_AGING) STUDY_TYPE else file.path(STUDY_TYPE, DISEASE)


# ─────────────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────────────
SCRATCH <- SCRATCH

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = file.path(REF_DIR, "markers"),
    sloan_xlsx     = file.path(REF_DIR, "markers", "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(REF_DIR, "markers", "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(REF_DIR, "markers", "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Sloan Table S9 column mapping — 10 senescence gene lists
# Order is FIXED (used as canonical row order in §5 forest plots):
#   rows 1-8: individual hallmarks
#   rows 9-10: multi-hallmark composites
# ─────────────────────────────────────────────────────────────────────────────
SLOAN_HALLMARK_NAMES <- c(
    "p53_Targets",
    "CellCycleArrest",
    "SASP",
    "AntiApoptosis",
    "DDR",
    "CellSurfaceMarkers",
    "LysosomalContent",
    "SD_TMC",
    "SenMayo",
    "Fridman_Up"
)

SLOAN_LIST_TYPES <- c(
    rep("Individual hallmark", 7),
    rep("Multi-hallmark", 3)
)

# Color labels for module rows in §5 forests:
# hallmarks = dark gray, multi-hallmark composites = muted purple
SLOAN_HALLMARK_COLORS <- c(
    rep("#222222", 8),
    rep("#7B5BA3", 2)
)
names(SLOAN_HALLMARK_COLORS) <- SLOAN_HALLMARK_NAMES


# ─────────────────────────────────────────────────────────────────────────────
# Proliferation markers (M09 §6 → M05 §7)
# ─────────────────────────────────────────────────────────────────────────────
PROLIFERATION_MARKERS <- c(  
  # Core proliferation / mitotic markers
  "MKI67", "TOP2A", "HMGB2", "CENPF",
  "BIRC5", "CCNB1", "CCNB2", "UBE2C",

  # Canonical CDK inhibitors
  "CDKN2A",  # p16INK4A
  "CDKN1A",  # p21CIP1/WAF1
  "CDKN1B",  # p27KIP1
  "CDKN2B",  # p15INK4B

  # p53 pathway / DNA damage response
  "TP53",
  "GADD45A", "GADD45B", "GADD45G"
)


# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# EFFECT_CONFIGS — display rules for forest plots, dispatched by beta_scale
#
# Each model cell selects one of these configs to drive the inline forest
# plot. No hardcoding of axis units, label formatting, or null reference
# value across §4.x or §5.x cells.
#
#   probability_pts:  paired-difference β on the proportion scale (used by
#                     §4.x cell-cycle phase analyses).
#   score_units:      continuous module-score β (used by §5.x Sloan module
#                     score analyses, including lmer Gaussian).
#   log_odds:         log-odds β (binomial GLMM, used by §4.4 only).
#                     - log-scale x-axis showing OR
#                     - effect column shows "OR = 1.38"
#                     - null line at OR = 1
# ─────────────────────────────────────────────────────────────────────────────
EFFECT_CONFIGS <- list(
    probability_pts = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (paired difference, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    score_units = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (mean module score, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    log_odds = list(
        scale          = "log",
        null_value     = 1,
        x_label        = "Odds Ratio (SnC vs Non-SnC)",
        eff_h_label    = "OR",
        ci_h_label     = "95% CI (OR)",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%.2f", exp(v))
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%.2f, %.2f]", exp(lo), exp(hi))
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) exp(v)
    )
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}


# =============================================================================
# CROSS-CELL HELPERS
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)


# ─────────────────────────────────────────────────────────────────────────────
# Summary banner
# ─────────────────────────────────────────────────────────────────────────────
cat("\n  Run parameters:\n")
cat(sprintf("    TISSUE         : %s\n", TISSUE))
cat(sprintf("    STUDY_TYPE     : %s\n", STUDY_TYPE))
if (IS_DISEASE)
    cat(sprintf("    DISEASE        : %s\n", DISEASE))
cat(sprintf("    DATASET        : %s\n", DATASET))
cat(sprintf("    CELL_TYPE      : %s\n", CELL_TYPE))
cat(sprintf("    CONDITION_TAG  : %s\n", CONDITION_TAG))

cat(sprintf("\n  Inline plot viewport: %dx%d\n", 10, 5))

cat("\n  Stratification:\n")
cat(sprintf("    STRATIFY_BY_GROUP     : %s\n", STRATIFY_BY_GROUP))
if (STRATIFY_BY_GROUP) {
    cat(sprintf("    STRATIFICATION_GROUPS : %s\n",
                paste(STRATIFICATION_GROUPS, collapse = ", ")))
    cat(sprintf("    Total runs per cell   : 1 (\"All\") + %d strata = %d\n",
                length(STRATIFICATION_GROUPS), 1 + length(STRATIFICATION_GROUPS)))
} else {
    cat("    Stratified runs       : disabled (overall only)\n")
}

cat("\n  Statistical params:\n")
cat(sprintf("    min_cells_per_group : %d\n", STATISTICAL_PARAMS$min_cells_per_group))
cat(sprintf("    fdr_threshold       : %.2f (%s)\n",
            STATISTICAL_PARAMS$fdr_threshold, STATISTICAL_PARAMS$fdr_method))
cat(sprintf("    bootstrap_n_iter    : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter, STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("    confidence_level    : %.2f\n", STATISTICAL_PARAMS$confidence_level))

cat("\n  Effect display configs:\n")
for (nm in names(EFFECT_CONFIGS)) {
    cfg <- EFFECT_CONFIGS[[nm]]
    cat(sprintf("    %-15s -> axis=%s, null=%g, label='%s'\n",
                nm, cfg$scale, cfg$null_value, cfg$x_label))
}

cat("\n  Cross-cell helpers (defined in §0):\n")
cat("    Formatting   : fmt_n, fmt_size, fmt_pct, fmt_elapsed, fmt_p, fmt_p_short, sig_stars\n")
cat("    Time/save    : now_iso, bytes_str, time_step, save_figure, save_table\n")
cat("    Color        : text_color_for_bg\n")
cat("    Stratum      : filter_to_stratum\n")
cat("    Results      : tidy_model_results\n")
cat("    (Donor data helpers: build_donor_arms, build_donor_meta, tidy_lmm_term -- defined in §3.6)\n")

cat("\n  Senescence gene lists:\n")
cat(sprintf("    Source              : Sloan Table S9 (sheet 'Sen Gene Lists')\n"))
cat(sprintf("    Lists               : %d (8 hallmarks + 2 multi-hallmark)\n",
            length(SLOAN_HALLMARK_NAMES)))
cat(sprintf("    Fixed row order     : %s\n",
            paste(SLOAN_HALLMARK_NAMES, collapse = ", ")))

cat("\n  Proliferation markers (§7):\n")
cat(sprintf("    %s\n", paste(PROLIFERATION_MARKERS, collapse = ", ")))

cat("\n  Library versions:\n")
for (pkg in c("R", "Seurat", "lme4", "robustbase", "broom.mixed",
              "qs", "jsonlite")) {
    cat(sprintf("    %-12s : %s\n", pkg, R_PKG_VERSIONS[[pkg]]))
}

cat("\n  Paths:\n")
cat(sprintf("    M04 input root     : %s\n", PATHS$m04_root))
cat(sprintf("    M05 output root    : %s\n", PATHS$output_root))

cat("\n  Required input files:\n")
for (key in c("m04_seurat", "m04_manifest", "sloan_xlsx",
              "senmayo_xlsx", "fridman_gmt")) {
    exists_flag <- if (file.exists(PATHS[[key]])) "✓" else "✗"
    sz <- if (file.exists(PATHS[[key]])) fmt_size(PATHS[[key]]) else "MISSING"
    cat(sprintf("    %s  %-15s : %s  (%s)\n",
                exists_flag, key, PATHS[[key]], sz))
}

cat("\n=", strrep("=", 71), "\n", sep = "")
cat("✓ §0 config loaded. All cross-cell helpers in scope.\n")
cat("  Inline viewport: 10x5 (set via options(repr.plot.*)).\n")
cat("  Run §1 to load M04 manifest + .qs and validate.\n")

# =============================================================================
# §0.2 — AXIS SELECT
# =============================================================================
# The ONE thing you change to re-run the whole flow on a different state.
# Everything downstream — column names, quadrant labels, figure titles,
# legends, output paths — derives from this. Nothing is hardcoded per state.
# =============================================================================

AXIS <- "IRM"          # <<< "IRM" | "DAM_like" | "ARM" | "Stress"

# --- registry: score column candidates + display label + canonical hex -------
# Score columns are looked up in order; the first present on the object wins.
AXIS_REGISTRY <- list(
    # tag = short token used in CONTRAST NAMES and therefore in GSEA/DE filenames.
    # It is deliberately NOT the same as `lab` (display) or the list key: existing
    # results on disk are named DAMaxis_*, not DAM_likeaxis_*.
    IRM      = list(cols = c("Score_IRM",      "IRM1"),      lab = "IRM",      tag = "IRM",    hex = "#2980B9"),
    DAM_like = list(cols = c("Score_DAM_like", "DAM_like1"), lab = "DAM-like", tag = "DAM",    hex = "#C0392B"),
    ARM      = list(cols = c("Score_ARM",      "ARM1"),      lab = "ARM",      tag = "ARM",    hex = "#E67E22"),
    Stress   = list(cols = c("Score_Stress",   "Stress1"),   lab = "Stress",   tag = "Stress", hex = "#8E44AD")
)
stopifnot(AXIS %in% names(AXIS_REGISTRY))

AXIS_SPEC <- AXIS_REGISTRY[[AXIS]]
X_LAB     <- AXIS_SPEC$lab
X_HEX     <- AXIS_SPEC$hex
Y_LAB     <- "Senescence"
Y_TAG     <- "SnC"
X_TAG     <- AXIS_SPEC$tag
SEN_CANDIDATES <- c("senescence_score", "SenePy_score")

# --- quadrant labels derive from the axis -----------------------------------
QUAD_COL    <- paste0("quad4_", tolower(AXIS))
QUAD_LEVELS <- c(sprintf("Sen- %s-", X_LAB), sprintf("Sen+ %s-", X_LAB),
                 sprintf("Sen- %s+", X_LAB), sprintf("Sen+ %s+", X_LAB))
QUAD_COLORS <- setNames(c("#B8B8B8", "#2E7D32", X_HEX, "#6A1B9A"), QUAD_LEVELS)

# --- the four DE / GSEA contrasts, named off the tags ------------------------
CONTRASTS <- c(sprintf("%saxis_%spos", X_TAG, Y_TAG),   # axis effect within SnC+
               sprintf("%saxis_%sneg", X_TAG, Y_TAG),   # axis effect within SnC-
               sprintf("%saxis_%spos", Y_TAG, X_TAG),   # sen effect within axis+
               sprintf("%saxis_%sneg", Y_TAG, X_TAG))   # sen effect within axis-
CONTRAST_HEADERS <- c(
    sprintf("%s+%s+ vs %s+%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s−%s+ vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s+ vs %s−%s+", Y_TAG, X_LAB, Y_TAG, X_LAB),
    sprintf("%s+%s− vs %s−%s−", Y_TAG, X_LAB, Y_TAG, X_LAB))

# --- outputs are namespaced by axis so runs never overwrite each other -------
AXIS_FIG_DIR <- file.path(PATHS$figures, AXIS)
AXIS_RES_DIR <- file.path(PATHS$results, AXIS)
dir.create(AXIS_FIG_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(AXIS_RES_DIR, recursive = TRUE, showWarnings = FALSE)

# --- state annotation (all five; NOT gated by AXIS) --------------------------
STATE_ORDER  <- c("Homeostatic", "ARM", "IRM", "Stress", "DAM_like")
STATE_COLORS <- c(Homeostatic = "#7F8C8D", ARM = "#E67E22", IRM = "#2980B9",
                  Stress = "#8E44AD", DAM_like = "#C0392B")

# --- DE model design (confirmed 2026-07-28; supersedes the leaner ~pop+grp2+Sex)
DE_DESIGN     <- "~ 0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort"
DE_COEF       <- "popTEST"
DE_MIN_CELLS  <- 10L

# --- GSEA (consumed by the python notebook via gsea_config.json) -------------
GSEA_DBS      <- c("Reactome_2022")
GSEA_FDR_SIG  <- 0.05
GSEA_N_COMMON <- 6L      # sig in >=3 contrasts, top N by mean NES
GSEA_N_UNIQUE <- 4L      # sig in exactly 1 contrast, top N by |NES|
GSEA_RIBO_STRIP <- FALSE # keep translational terms; see Part F Why



# §0.3 — AXIS-AWARE HELPERS  (one definition each — see note)
# =============================================================================
# In the source notebook fit_one was defined 13x, resolve_score_col 6x,
# z_score 5x, venn2 4x, and save_figure was REDEFINED at cells 317/341,
# shadowing the canonical version above. Everything lives here now so a
# later cell cannot silently shadow it.
# =============================================================================

# --- resolve a score column from candidates ---------------------------------
resolve_score_col <- function(md, candidates, what = "score") {
    hit <- candidates[candidates %in% colnames(md)]
    if (!length(hit)) stop(sprintf("no %s column found; tried: %s",
                                   what, paste(candidates, collapse = ", ")))
    hit[1]
}

z_score <- function(x) as.numeric(scale(x))

# --- build the SnC x AXIS quadrant column -----------------------------------
# Mean-split on z-scored values, exactly as the source (cell 54).
build_quadrants <- function(obj, axis = AXIS, verbose = TRUE) {
    md   <- obj@meta.data
    spec <- AXIS_REGISTRY[[axis]]
    sen  <- resolve_score_col(md, SEN_CANDIDATES, "senescence")
    xcol <- resolve_score_col(md, spec$cols, paste(axis, "score"))
    sz <- z_score(md[[sen]]); xz <- z_score(md[[xcol]])
    lab <- spec$lab
    q <- ifelse(sz >  0 & xz >  0, sprintf("Sen+ %s+", lab),
        ifelse(sz >  0 & xz <= 0, sprintf("Sen+ %s-", lab),
        ifelse(sz <= 0 & xz >  0, sprintf("Sen- %s+", lab),
                                  sprintf("Sen- %s-", lab))))
    q[is.na(sz) | is.na(xz)] <- NA
    obj[[paste0("quad4_", tolower(axis))]] <- q
    obj$sen_z <- sz
    obj$axis_z <- xz
    if (verbose) {
        cat(sprintf("  scores : sen=%s  %s=%s\n", sen, axis, xcol))
        print(table(q, useNA = "ifany"))
        cat(sprintf("  cor(sen_z, %s_z) = %.3f   <- independence check\n",
                    tolower(axis), cor(sz, xz, use = "complete.obs")))
    }
    obj
}

# --- PREFLIGHT: fail loudly before any model runs ---------------------------
preflight_axis <- function(obj, axis = AXIS, min_cells = 50L, donor_col = "Donor") {
    md <- obj@meta.data; ok <- TRUE
    say <- function(pass, msg) {
        cat(sprintf("  [%s] %s\n", if (pass) "OK  " else "FAIL", msg))
        if (!pass) ok <<- FALSE
    }
    cat(sprintf("\n── PREFLIGHT · axis = %s ──\n", axis))
    say(axis %in% names(AXIS_REGISTRY), sprintf("axis '%s' is registered", axis))
    spec <- AXIS_REGISTRY[[axis]]
    xhit <- spec$cols[spec$cols %in% colnames(md)]
    say(length(xhit) > 0, sprintf("score column present (%s)",
        if (length(xhit)) xhit[1] else paste(spec$cols, collapse = "/")))
    shit <- SEN_CANDIDATES[SEN_CANDIDATES %in% colnames(md)]
    say(length(shit) > 0, "senescence score column present")
    if (length(xhit) && length(shit)) {
        say(sd(md[[xhit[1]]], na.rm = TRUE) > 0, "axis score is non-constant")
        say(sd(md[[shit[1]]], na.rm = TRUE) > 0, "senescence score is non-constant")
    }
    qc <- paste0("quad4_", tolower(axis))
    if (qc %in% colnames(md)) {
        tb <- table(md[[qc]])
        say(length(tb) == 4, sprintf("all four quadrants populated (%d)", length(tb)))
        say(all(tb >= min_cells), sprintf("every quadrant >= %d cells (min %d)",
                                          min_cells, min(tb)))
        if (donor_col %in% colnames(md)) {
            nd <- tapply(md[[donor_col]], md[[qc]], function(z) length(unique(z)))
            say(all(nd >= 2), sprintf("every quadrant has >=2 donors (min %d)", min(nd)))
        }
    } else cat(sprintf("  [--  ] %s not built yet (run B1)\n", qc))
    cat(sprintf("── %s ──\n\n", if (ok) "PASS" else "STOP: fix before proceeding"))
    invisible(ok)
}

# --- ONE mixed-model fitter (replaces 13 copies of fit_one) -----------------
# formula_str is built by the caller, so every Part C analysis is this
# function with a different formula and a different subset.
fit_lmm <- function(df, formula_str, term, label = NA_character_) {
    fit <- tryCatch(lmerTest::lmer(as.formula(formula_str), data = df,
                                   REML = TRUE,
                                   control = lme4::lmerControl(
                                       optimizer = "bobyqa",
                                       optCtrl = list(maxfun = 2e5))),
                    error = function(e) NULL, warning = function(w) NULL)
    if (is.null(fit)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    co <- summary(fit)$coefficients
    if (!term %in% rownames(co)) return(data.frame(label = label, term = term,
                                        beta = NA, se = NA, ci_low = NA,
                                        ci_high = NA, p_value = NA,
                                        n = nrow(df), converged = FALSE))
    b <- co[term, "Estimate"]; s <- co[term, "Std. Error"]
    data.frame(label = label, term = term, beta = b, se = s,
               ci_low = b - 1.96 * s, ci_high = b + 1.96 * s,
               p_value = co[term, "Pr(>|t|)"], n = nrow(df), converged = TRUE)
}

# --- ONE forest renderer (replaces 7 near-copies) ---------------------------
forest_plot <- function(res, title = "", xlab = "beta (95% CI)",
                        facet = NULL, color = X_HEX) {
    stopifnot(all(c("label", "beta", "ci_low", "ci_high") %in% names(res)))
    if (!"p_adj" %in% names(res))
        res$p_adj <- p.adjust(res$p_value, method = STATISTICAL_PARAMS$fdr_method)
    res$sig <- sig_stars(res$p_adj)
    res$label <- factor(res$label, levels = rev(unique(res$label)))
    p <- ggplot(res, aes(x = beta, y = label)) +
        geom_vline(xintercept = 0, linetype = "dashed",
                   colour = "grey60", linewidth = 0.3) +
        geom_errorbarh(aes(xmin = ci_low, xmax = ci_high),
                       height = 0, linewidth = 0.4, colour = color) +
        geom_point(size = 1.8, colour = color) +
        geom_text(aes(x = ci_high, label = sig), hjust = -0.35,
                  size = 2.6, na.rm = TRUE) +
        labs(title = title, x = xlab, y = NULL) +
        theme_clean() +
        theme(panel.grid.major.y = element_line(colour = "grey92", linewidth = 0.25))
    if (!is.null(facet)) p <- p + facet_wrap(as.formula(paste("~", facet)), scales = "free_x")
    p + coord_cartesian(clip = "off")
}

# --- block banner: prints the Why with the axis resolved --------------------
say_block <- function(id, title, why = NULL) {
    cat("\n", strrep("═", 76), "\n", sep = "")
    cat(sprintf("%s  ·  %s\n", id, sprintf(title, X_LAB)))
    cat(strrep("═", 76), "\n", sep = "")
    if (!is.null(why)) cat(sprintf("WHY: %s\n\n", sprintf(why, X_LAB)))
}

# X_COL is resolved once the object exists, in the load section:
#     X_COL <- resolve_score_col(mg@meta.data, AXIS_SPEC$cols, 'axis')
# Every downstream cell reads X_COL, never a literal score column.

---
## 02 · Gene-set inventory

**Why.** Locates the stored gene sets and reports what is present before any model runs, so a missing panel fails here rather than silently producing an empty enrichment later.

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) -> X_COL,
#       resolved from AXIS in the config cell.
library(dplyr)
md <- obj_ct@meta.data
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")

df <- data.frame(
    sen  = md$senescence_score,   # SenePy
    sasp = md$Score_SASP,
    dam  = md[[X_COL]],
    grp  = gmap[as.character(md$Study_Group)]
)
df <- df[is.finite(df$sen) & is.finite(df$sasp) & is.finite(df$dam) & !is.na(df$grp), ]

pair_cor <- function(d, a, b) {
    data.frame(pair=sprintf("%s vs %s", a, b),
               pearson = cor(d[[a]], d[[b]], method="pearson"),
               spearman= cor(d[[a]], d[[b]], method="spearman"))
}

cat("=", strrep("=",58), "\n", sep="")
cat("Score correlations (microglia)\n")
cat("=", strrep("=",58), "\n", sep="")

for (g in c("All","Control","AD")) {
    sub <- if (g=="All") df else df[df$grp==g, ]
    cat(sprintf("\n── %s (n=%d) ──\n", g, nrow(sub)))
    res <- rbind(
        pair_cor(sub, "sen",  "dam"),    # SenePy vs DAM  (expect ~0)
        pair_cor(sub, "sasp", "dam"),    # SASP   vs DAM  (expect high +)
        pair_cor(sub, "sen",  "sasp")    # SenePy vs SASP (the key one)
    )
    for (i in seq_len(nrow(res)))
        cat(sprintf("  %-14s  Pearson r=%+.3f   Spearman=%+.3f\n",
                    res$pair[i], res$pearson[i], res$spearman[i]))
}
cat("\n  Interpretation guide:\n")
cat("   sasp~dam high + → SASP axis is DAM-collinear (quadrant breaks)\n")
cat("   sen~dam  ~0     → SenePy axis IS independent of DAM (quadrant OK)\n")
cat("   sen~sasp        → whether SenePy and SASP even agree\n")

In [ ]:
# look for stored gene sets in the Seurat object
cat("obj_ct@misc names:\n"); print(names(obj_ct@misc))
cat("\nobj_ct@tools names:\n"); print(names(obj_ct@tools))

# common stash patterns
for (slot in c("misc","tools")) {
    x <- slot(obj_ct, slot)
    if (length(x)) for (nm in names(x)) {
        if (grepl("sig|gene|module|score|panel|set", nm, ignore.case=TRUE))
            cat(sprintf("\n[%s$%s] class=%s\n", slot, nm, class(x[[nm]])[1]))
    }
}

# also scan the global env for a gene-list object
gl <- ls(envir=.GlobalEnv)
cat("\nGlobal objects matching sig/gene/module/panel:\n")
print(grep("sig|gene|module|panel|score|SASP|DAM|signature", gl, value=TRUE, ignore.case=TRUE))

In [ ]:
# inspect the gene-list objects
cat("=== gene_lists ===\n")
cat("class:", class(gene_lists), "| length:", length(gene_lists), "\n")
cat("names:\n"); print(names(gene_lists))
if (length(gene_lists)) { cat("\nexample (first entry):\n"); print(head(gene_lists[[1]], 10)) }

cat("\n=== MICROGLIA_STATE_PANELS ===\n")
cat("class:", class(MICROGLIA_STATE_PANELS), "| length:", length(MICROGLIA_STATE_PANELS), "\n")
print(names(MICROGLIA_STATE_PANELS))
if (length(MICROGLIA_STATE_PANELS)) print(head(MICROGLIA_STATE_PANELS[[1]], 10))

cat("\n=== module_registry ===\n")
cat("class:", class(module_registry), "\n")
if (is.list(module_registry)) print(names(module_registry)) else print(head(module_registry))

cat("\n=== gene_lists_info ===\n")
print(gene_lists_info)

In [ ]:
library(dplyr)

# combine all panels into one named list
all_panels <- c(gene_lists, MICROGLIA_STATE_PANELS)
# clean: uppercase, unique, drop empties
all_panels <- lapply(all_panels, function(g) unique(toupper(as.character(g))))
pn <- names(all_panels)

cat("Panel sizes:\n")
for (nm in pn) cat(sprintf("  %-20s %d genes\n", nm, length(all_panels[[nm]])))

# ── pairwise overlap: shared count + Jaccard ────────────────────────────────
K <- length(all_panels)
shared <- matrix(0, K, K, dimnames=list(pn,pn))
jacc   <- matrix(0, K, K, dimnames=list(pn,pn))
for (i in 1:K) for (j in 1:K) {
    a <- all_panels[[i]]; b <- all_panels[[j]]
    inter <- length(intersect(a,b)); uni <- length(union(a,b))
    shared[i,j] <- inter
    jacc[i,j]   <- if (uni>0) inter/uni else 0
}

cat("\n=== SASP overlap with every other panel ===\n")
cat(sprintf("  SASP has %d genes\n\n", length(all_panels[["SASP"]])))
ord <- order(-shared["SASP",])
for (nm in pn[ord]) {
    if (nm=="SASP") next
    sh <- shared["SASP",nm]; jc <- jacc["SASP",nm]
    if (sh>0) {
        common <- intersect(all_panels[["SASP"]], all_panels[[nm]])
        cat(sprintf("  SASP ∩ %-18s : %2d shared (Jaccard %.3f)  {%s}\n",
                    nm, sh, jc, paste(head(common,12), collapse=", ")))
    } else cat(sprintf("  SASP ∩ %-18s : 0 shared\n", nm))
}

# ── specifically SASP vs DAM_like (the r=0.35 pair) ─────────────────────────
cat("\n=== SASP ∩ DAM_like (the correlation culprit) ===\n")
common_sd <- intersect(all_panels[["SASP"]], all_panels[["DAM_like"]])
cat(sprintf("  shared genes (%d): %s\n", length(common_sd),
            paste(common_sd, collapse=", ")))
cat(sprintf("  SASP unique: %d | DAM_like unique: %d\n",
            length(setdiff(all_panels[["SASP"]], all_panels[["DAM_like"]])),
            length(setdiff(all_panels[["DAM_like"]], all_panels[["SASP"]]))))

# save overlap matrices
save_table(as.data.frame(shared) %>% tibble::rownames_to_column("panel"),
           "panel_overlap_shared_counts")
save_table(as.data.frame(round(jacc,3)) %>% tibble::rownames_to_column("panel"),
           "panel_overlap_jaccard")
cat("\n✓ overlap computed.\n")

---
## 03 · Pseudobulk build

**Why.** Seven steps, each with a check, because a silent failure here propagates into every contrast.

1. quadrant labels and group mapping — counts sanity-checked
2. donor-level cohort structure — decides whether cohorts pool
3. contrast setup with an explicit pairing check
4. aggregation, with per-sample cell counts merged **before** filtering so the filter can be seen to do what it claims
5. minimum-cell filter, then **re-pair** — dropping a thin sample can leave its donor unpaired, so pairing is recomputed after filtering
6. design matrix, with the contrast verified rather than assumed
7. voom + donor-blocked limma on the single contrast, as a check before the loop

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1a — define pop2 (two quadrants on a Y×X score pair) + subset
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
OBJ_IN <- obj_ct
Y_COL  <- "senescence_score"; Y_TAG <- "Sen"   # the axis that varies (Hi vs Lo)
X_COL  <- "Score_IRM";   X_TAG <- "IRM"   # the axis held LOW in both pops
GROUPS <- c("Old_AD"="AD", "Old_Healthy_Control"="Control")
# ════════════════════════════════════════════════════════════════════════════

OBJ <- OBJ_IN
md  <- OBJ@meta.data
stopifnot(Y_COL %in% colnames(md), X_COL %in% colnames(md))

y_z <- as.numeric(scale(md[[Y_COL]]))
x_z <- as.numeric(scale(md[[X_COL]]))

POP_HI <- sprintf("%sHi_%slo", Y_TAG, X_TAG)   # Y high, X low
POP_LO <- sprintf("%sLo_%slo", Y_TAG, X_TAG)   # Y low,  X low
OBJ$pop2 <- ifelse(y_z >  0 & x_z <= 0, POP_HI,
            ifelse(y_z <= 0 & x_z <= 0, POP_LO, NA))
OBJ$grp2 <- unname(GROUPS[as.character(md$Study_Group)])

cat(strrep("=",80), "\n", sep="")
cat("STEP 1a — populations defined  (", Y_TAG, " split within ", X_TAG, "-low)\n", sep="")
cat(strrep("=",80), "\n", sep="")
cat(sprintf("\nTotal microglia: %d\n", ncol(OBJ)))
cat("\npop2 (before subsetting):\n"); print(table(OBJ$pop2, useNA="ifany"))
cat("\ngrp2:\n"); print(table(OBJ$grp2, useNA="ifany"))

keep <- !is.na(OBJ$pop2) & !is.na(OBJ$grp2)
OBJ  <- OBJ[, keep]
cat(sprintf("\nAfter subsetting: %d cells\n", ncol(OBJ)))
cat("\npop2 \u00d7 grp2 (cells):\n")
print(table(OBJ$pop2, OBJ$grp2))

seurat_pb <- OBJ
cat(sprintf("\n\u2713 step 1a done \u2014 `seurat_pb` holds %s vs %s.\n", POP_HI, POP_LO))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1b — single-cell covariates before aggregation
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr) })
if (!exists("seurat_pb")) stop("✗ run step 1a first")

DONOR_COL <- "Donor"; SEX_COL <- "Sex"; AGE_COL <- "Age"; COHORT_COL <- "Cohort"

# ensure nCount_RNA
if (!"nCount_RNA" %in% colnames(seurat_pb@meta.data)) {
    seurat_pb$nCount_RNA <- colSums(GetAssayData(seurat_pb, slot="counts"))
    cat("  ✓ computed nCount_RNA from counts\n")
} else cat("  ✓ nCount_RNA present\n")

HAS_COHORT <- COHORT_COL %in% colnames(seurat_pb@meta.data)
cat(sprintf("  HAS_COHORT: %s\n", HAS_COHORT))

# grouping columns for pseudobulk (donor × group × pop × sex [× cohort])
group_by_cols <- c(DONOR_COL, "grp2", "pop2", SEX_COL)
if (HAS_COHORT) group_by_cols <- c(group_by_cols, COHORT_COL)
cat(sprintf("  grouping by: %s\n", paste(group_by_cols, collapse=", ")))

# donor-level mean covariates (computed from single cells, per reference)
donor_covs <- seurat_pb@meta.data %>%
    group_by(across(all_of(group_by_cols))) %>%
    summarise(
        Mean_Age = mean(as.numeric(.data[[AGE_COL]]), na.rm=TRUE),
        Mean_Log_Library_Depth = mean(log10(nCount_RNA + 1), na.rm=TRUE),
        n_cells = n(),
        .groups="drop"
    ) %>% as.data.frame()

cat(sprintf("\n  ✓ covariates for %d donor×group×pop combinations\n", nrow(donor_covs)))
cat(sprintf("    Mean_Age range: %.1f – %.1f\n",
            min(donor_covs$Mean_Age), max(donor_covs$Mean_Age)))
cat(sprintf("    Mean_Log_Library_Depth range: %.3f – %.3f\n",
            min(donor_covs$Mean_Log_Library_Depth), max(donor_covs$Mean_Log_Library_Depth)))
cat(sprintf("    n_cells/combo: median %.0f, min %d, max %d\n",
            median(donor_covs$n_cells), min(donor_covs$n_cells), max(donor_covs$n_cells)))

cat("\n✓ step 1b done — donor_covs + group_by_cols ready.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1c — AggregateExpression (pseudobulk)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat) })
if (!exists("donor_covs")) stop("✗ run step 1b first")

cat("--- Aggregating expression ---\n")
cat(sprintf("  grouping by: %s\n", paste(group_by_cols, collapse=", ")))

pseudo <- AggregateExpression(
    seurat_pb,
    assays = "RNA",
    return.seurat = TRUE,
    group.by = group_by_cols
)

cat(sprintf("\n✓ Pseudobulk Seurat: %d samples × %d genes\n",
            ncol(pseudo), nrow(pseudo)))

cat("\nSample names (first 5):\n")
print(head(Cells(pseudo), 5))

cat("\npop2 levels as stored in pseudobulk (note _ may become -):\n")
print(table(pseudo[["pop2", drop=TRUE]]))

cat("\ngrp2 levels:\n")
print(table(pseudo[["grp2", drop=TRUE]]))

cat("\n✓ step 1c done — `pseudo` created.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1d — keep paired donors (both pops) + attach covariates
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr) })
if (!exists("pseudo")) stop("\u2717 run step 1c first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
DONOR_COL <- "Donor"
Y_TAG <- "Sen"; X_TAG <- "IRM"                      # must match STEP 1a
HI <- sprintf("%sHi-%slo", Y_TAG, X_TAG)            # e.g. "SenHi-DAMlo"
LO <- sprintf("%sLo-%slo", Y_TAG, X_TAG)            # e.g. "SenLo-DAMlo"
COVARS_NUM <- c("Mean_Age","Mean_Log_Library_Depth")  # numeric covariates to impute/attach
# ════════════════════════════════════════════════════════════════════════════

# ── paired-donor filter: keep donors with BOTH pops ─────────────────────────
pb_donor <- pseudo[[DONOR_COL, drop=TRUE]]
pb_pop   <- pseudo[["pop2", drop=TRUE]]
tab <- table(pb_donor, pb_pop)
stopifnot(all(c(HI, LO) %in% colnames(tab)))
paired <- rownames(tab)[tab[, HI] > 0 & tab[, LO] > 0]
n_before <- ncol(pseudo)
pseudo <- pseudo[, pb_donor %in% paired]
cat(sprintf("  contrast: %s vs %s\n", HI, LO))
cat(sprintf("  paired donors (both pops): %d / %d\n", length(paired), length(unique(pb_donor))))
cat(sprintf("  samples retained: %d / %d (dropped %d unpaired)\n",
            ncol(pseudo), n_before, n_before - ncol(pseudo)))

# ── attach covariates: underscore→dash on donor_covs keys to match ──────────
for (col in group_by_cols)
    if (is.character(donor_covs[[col]]) || is.factor(donor_covs[[col]]))
        donor_covs[[col]] <- gsub("_", "-", as.character(donor_covs[[col]]))
pb_meta <- pseudo@meta.data; pb_meta$row_id <- rownames(pb_meta)
for (col in group_by_cols) pb_meta[[col]] <- as.character(pb_meta[[col]])
merged <- merge(pb_meta, donor_covs, by=group_by_cols, all.x=TRUE, sort=FALSE)
merged <- merged[match(pb_meta$row_id, merged$row_id), ]

n_na <- sum(sapply(COVARS_NUM, function(c) sum(is.na(merged[[c]]))))
if (n_na > 0) {
    cat(sprintf("  \u26a0 %d covariate NAs \u2014 imputing with median\n", n_na))
    for (c in COVARS_NUM)
        merged[[c]][is.na(merged[[c]])] <- median(merged[[c]], na.rm=TRUE)
} else cat(sprintf("  \u2713 all %d samples matched covariates \u2014 no NAs\n", nrow(merged)))

for (c in COVARS_NUM) pseudo[[c]] <- merged[[c]]
pseudo[["Log_nCells"]] <- log10(merged$n_cells)
pseudo[["n_cells"]]    <- merged$n_cells

cat("\nSamples per pop \u00d7 group (after pairing):\n")
print(table(pseudo[["pop2", drop=TRUE]], pseudo[["grp2", drop=TRUE]]))
cat(sprintf("\n  cells/sample: median %.0f, min %d (watch <10)\n",
            median(merged$n_cells), min(merged$n_cells)))
cat("\n\u2713 step 1d done \u2014 covariates attached, paired samples only.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1e — filter pseudobulk samples by min cells, re-enforce pairing
# ════════════════════════════════════════════════════════════════════════════
if (!exists("pseudo")) stop("\u2717 run step 1d first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
DONOR_COL <- "Donor"; MIN_CELLS <- 10
Y_TAG <- "Sen"; X_TAG <- "IRM"                 # must match STEP 1a / 1d
HI <- sprintf("%sHi-%slo", Y_TAG, X_TAG)
LO <- sprintf("%sLo-%slo", Y_TAG, X_TAG)
# ════════════════════════════════════════════════════════════════════════════

n_before <- ncol(pseudo)
nc <- pseudo[["n_cells", drop=TRUE]]
cat(sprintf("Samples with <%d cells: %d\n", MIN_CELLS, sum(nc < MIN_CELLS)))

# 1) drop thin samples
pseudo <- pseudo[, nc >= MIN_CELLS]
cat(sprintf("  after cell-count filter: %d / %d samples\n", ncol(pseudo), n_before))

# 2) re-enforce pairing (a donor may have lost its partner)
pb_donor <- pseudo[[DONOR_COL, drop=TRUE]]; pb_pop <- pseudo[["pop2", drop=TRUE]]
tab <- table(pb_donor, pb_pop)
stopifnot(all(c(HI, LO) %in% colnames(tab)))
paired <- rownames(tab)[tab[, HI] > 0 & tab[, LO] > 0]
pseudo <- pseudo[, pb_donor %in% paired]
cat(sprintf("  contrast: %s vs %s\n", HI, LO))
cat(sprintf("  after re-pairing: %d donors, %d samples\n", length(paired), ncol(pseudo)))

cat("\nSamples per pop \u00d7 group (final):\n")
print(table(pseudo[["pop2", drop=TRUE]], pseudo[["grp2", drop=TRUE]]))
ncf <- pseudo[["n_cells", drop=TRUE]]
cat(sprintf("\n  cells/sample now: median %.0f, min %d, max %d\n",
            median(ncf), min(ncf), max(ncf)))
cat("\n\u2713 step 1e done \u2014 filtered, paired pseudobulk ready for limma.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1f — DGEList, gene filter, TMM normalization
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(edgeR); library(limma) })
if (!exists("pseudo")) stop("\u2717 run step 1e first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
DONOR_COL <- "Donor"; SEX_COL <- "Sex"; COHORT_COL <- "Cohort"
HAS_COHORT <- TRUE
Y_TAG <- "Sen"; X_TAG <- "IRM"                 # must match STEP 1a–1e
HI <- sprintf("%sHi-%slo", Y_TAG, X_TAG)       # pop2 value → mapped to "<Y_TAG>Hi"
# ════════════════════════════════════════════════════════════════════════════

# ── extract counts + metadata ────────────────────────────────────────────────
counts_mat <- as.matrix(GetAssayData(pseudo, layer="counts"))
meta_df    <- pseudo@meta.data

# types
meta_df[[SEX_COL]]   <- as.factor(meta_df[[SEX_COL]])
meta_df[[DONOR_COL]] <- as.factor(meta_df[[DONOR_COL]])
if (HAS_COHORT) {
    meta_df[[COHORT_COL]] <- as.factor(meta_df[[COHORT_COL]])
    levels(meta_df[[COHORT_COL]]) <- make.names(levels(meta_df[[COHORT_COL]]))
}

# contrast variable: pop2 (Hi vs Lo), + group covariate
HILAB <- sprintf("%sHi", Y_TAG); LOLAB <- sprintf("%sLo", Y_TAG)
meta_df$pop  <- factor(ifelse(meta_df$pop2==HI, HILAB, LOLAB), levels=c(LOLAB, HILAB))
meta_df$grp2 <- as.factor(meta_df$grp2)

# scaled continuous covariates
meta_df$Mean_Age_scaled               <- scale(meta_df$Mean_Age)[,1]
meta_df$Mean_Log_Library_Depth_scaled <- scale(meta_df$Mean_Log_Library_Depth)[,1]

cat(sprintf("\n\u2713 counts: %d genes \u00d7 %d samples\n", nrow(counts_mat), ncol(counts_mat)))
cat(sprintf("\u2713 pop levels: %s (ref=%s)\n", paste(levels(meta_df$pop), collapse=", "), LOLAB))
cat(sprintf("\u2713 donors: %d | groups: %s%s\n",
            nlevels(meta_df[[DONOR_COL]]), paste(levels(meta_df$grp2), collapse="/"),
            if (HAS_COHORT) sprintf(" | cohorts: %d", nlevels(meta_df[[COHORT_COL]])) else ""))

# ── DGEList + filter + TMM ───────────────────────────────────────────────────
dge  <- DGEList(counts=counts_mat, samples=meta_df, group=meta_df$pop)
keep <- filterByExpr(dge, group=meta_df$pop)
dge  <- dge[keep, , keep.lib.sizes=FALSE]
cat(sprintf("\n  genes retained: %d / %d\n", nrow(dge), nrow(counts_mat)))
dge <- calcNormFactors(dge, method="TMM")
cat("  \u2713 TMM normalization applied\n")
cat(sprintf("  library size range: %.2f \u2013 %.1f M\n",
            min(dge$samples$lib.size)/1e6, max(dge$samples$lib.size)/1e6))
cat("\n\u2713 step 1f done \u2014 `dge` and `meta_df` ready for voom.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1g — voom + donor-blocked limma, contrast SenHi - SenLo
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(limma); library(edgeR) })
if (!exists("dge")) stop("\u2717 run step 1f first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
DONOR_COL <- "Donor"; SEX_COL <- "Sex"; COHORT_COL <- "Cohort"
HAS_COHORT <- TRUE
Y_TAG <- "Sen"; X_TAG <- "IRM"                 # must match STEP 1a–1f
FDR_THRESHOLD <- 0.05; LFC_THRESHOLD <- 0
HILAB <- sprintf("%sHi", Y_TAG); LOLAB <- sprintf("%sLo", Y_TAG)
# ════════════════════════════════════════════════════════════════════════════

# ── design: ~0 + pop + grp2 + Sex + log-depth (+ Cohort)  (NO age: all Old) ──
design_str <- paste0("~ 0 + pop + grp2 + ", SEX_COL, " + Mean_Log_Library_Depth_scaled",
                     if (HAS_COHORT) paste0(" + ", COHORT_COL) else "")
cat("Design:", design_str, "\n")
design <- model.matrix(as.formula(design_str), data=meta_df)
colnames(design) <- make.names(colnames(design))
cat("Design columns:", paste(colnames(design), collapse=", "), "\n")
cat(sprintf("  rank: %d / %d cols\n", qr(design)$rank, ncol(design)))
stopifnot(qr(design)$rank == ncol(design))     # catch rank-deficiency before voom

# ── voom ─────────────────────────────────────────────────────────────────────
v <- voom(dge, design, plot=FALSE)

# ── donor blocking (paired: each donor has both pops) ───────────────────────
corfit <- duplicateCorrelation(v, design, block=meta_df[[DONOR_COL]])
cat(sprintf("\n  donor consensus correlation: %.3f\n", corfit$consensus.correlation))
fit <- lmFit(v, design, block=meta_df[[DONOR_COL]],
             correlation=corfit$consensus.correlation)

# ── contrast: Hi - Lo (built by name so it tracks Y_TAG) ─────────────────────
hi_col <- make.names(paste0("pop", HILAB)); lo_col <- make.names(paste0("pop", LOLAB))
stopifnot(hi_col %in% colnames(design), lo_col %in% colnames(design))
contrast_mat <- makeContrasts(contrasts=paste(hi_col, "-", lo_col), levels=design)
fit <- contrasts.fit(fit, contrast_mat)
fit <- eBayes(fit)

# ── results ──────────────────────────────────────────────────────────────────
res <- topTable(fit, coef=1, number=Inf, sort.by="P")
res$gene <- rownames(res)
res$direction <- ifelse(res$adj.P.Val < FDR_THRESHOLD & res$logFC >  LFC_THRESHOLD, "Up",
                 ifelse(res$adj.P.Val < FDR_THRESHOLD & res$logFC < -LFC_THRESHOLD, "Down", "NS"))
n_up <- sum(res$direction=="Up"); n_down <- sum(res$direction=="Down")
cat(sprintf("\n  DEGs (FDR<%.2f): %d up, %d down (%d total of %d tested)\n",
            FDR_THRESHOLD, n_up, n_down, n_up+n_down, nrow(res)))
cat("\n  Top 15 by p-value:\n")
print(head(res[, c("gene","logFC","P.Value","adj.P.Val","direction")], 15))
save_table(res, sprintf("pseudobulk_DE_%sHi%slo_vs_%sLo%slo_microglia", Y_TAG, X_TAG, Y_TAG, X_TAG))
cat("\n\u2713 step 1g done \u2014 `res` (full DE table) in scope.\n")

In [ ]:
# AXIS: 2 hardcoded 'Score_DAM_like' reference(s) -> X_COL,
#       resolved from AXIS in the config cell.
# ════════════════════════════════════════════════════════════════════════════
# STEP 1 — quadrant labels + group mapping · sanity check counts
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })
stopifnot(exists("obj_ct"))

md0 <- obj_ct@meta.data
stopifnot(all(c("senescence_score",X_COL,"Study_Group","Donor","Sex") %in% colnames(md0)))

# z-score the two axes (whole-population scaling, as before)
sen_z <- as.numeric(scale(md0$senescence_score))
dam_z <- as.numeric(scale(md0[[X_COL]]))

obj_ct$quad <- ifelse(sen_z>0  & dam_z>0,  "SnCpos_DAMpos",
               ifelse(sen_z>0  & dam_z<=0, "SnCpos_DAMneg",
               ifelse(sen_z<=0 & dam_z>0,  "SnCneg_DAMpos", "SnCneg_DAMneg")))

# disease group mapping
gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
obj_ct$grp2 <- unname(gmap[as.character(md0$Study_Group)])

cat("Study_Group values present:\n"); print(table(md0$Study_Group, useNA="ifany"))
cat("\ngrp2 mapping result:\n"); print(table(obj_ct$grp2, useNA="ifany"))
cat("\nCells per quadrant x group:\n")
print(table(obj_ct$quad, obj_ct$grp2, useNA="ifany"))
cat("\nCohort levels:\n"); print(table(obj_ct$Cohort, useNA="ifany"))
cat("\nSex levels:\n"); print(table(obj_ct$Sex, useNA="ifany"))
cat("\nDonors total:", length(unique(obj_ct$Donor)), "\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1b — donor-level cohort/group structure (decide cohort pooling)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr) })

# restrict to the Old cells we'll actually use (drop Young)
md <- obj_ct@meta.data %>% filter(!is.na(grp2))
cat("After dropping Young (NA grp2):", nrow(md), "cells |",
    length(unique(md$Donor)), "donors\n\n")

# donors per cohort
cat("Donors per Cohort (Old AD + Old HC only):\n")
donor_cohort <- md %>% distinct(Donor, Cohort, grp2)
print(table(donor_cohort$Cohort))

cat("\nCohort x group (donor counts) — check for confounding:\n")
print(table(donor_cohort$Cohort, donor_cohort$grp2))

cat("\nSex x group (donor counts):\n")
print(table(distinct(md, Donor, Sex, grp2)$Sex, distinct(md, Donor, Sex, grp2)$grp2))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 2 — single contrast setup (DAMaxis_SnCpos) · pop factor + pairing check
#   design will be ~ pop + grp2 + Sex (intercept; popTEST coef = TEST - REF)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

TEST <- "SnCpos_DAMpos"; REF <- "SnCpos_DAMneg"   # first contrast to validate

OBJ <- obj_ct[, obj_ct$quad %in% c(TEST,REF) & !is.na(obj_ct$grp2)]
OBJ$pop <- factor(ifelse(OBJ$quad==TEST, "TEST", "REF"), levels=c("REF","TEST"))  # REF = baseline
if (!"nCount_RNA" %in% colnames(OBJ@meta.data))
  OBJ$nCount_RNA <- colSums(GetAssayData(OBJ, slot="counts"))

cat("Contrast:", TEST, "(TEST) vs", REF, "(REF)\n")
cat("Cells:", ncol(OBJ), "| donors:", length(unique(OBJ$Donor)), "\n\n")

cat("pop x grp2 (cell counts):\n"); print(table(OBJ$pop, OBJ$grp2))
cat("\nDonor pairing — donors with BOTH pops (cell counts per donor):\n")
tab <- table(OBJ$Donor, OBJ$pop)
both <- rownames(tab)[tab[,"TEST"]>0 & tab[,"REF"]>0]
cat("  donors with both quadrants:", length(both), "of", nrow(tab), "\n")
cat("  donors missing one side:", nrow(tab)-length(both), "(will be dropped)\n")
cat("\n  distribution of per-donor cell counts (both pops):\n")
print(summary(as.vector(tab[both,])))
cat("\n  donors with <10 cells in either pop (will be dropped at min-cell step):\n")
thin <- both[apply(tab[both,], 1, min) < 10]
cat("  ", length(thin), "donors\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 3 — pseudobulk aggregation + n_cells merge (validate before filtering)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr) })

DONOR_COL<-"Donor"; SEX_COL<-"Sex"
gbc <- c(DONOR_COL, "grp2", "pop", SEX_COL)        # grouping for pseudobulk (no Cohort)
gbc <- gbc[gbc %in% colnames(OBJ@meta.data)]
cat("Pseudobulk grouping columns:", paste(gbc, collapse=", "), "\n\n")

# per-sample cell counts BEFORE aggregating
donor_covs <- OBJ@meta.data %>%
  group_by(across(all_of(gbc))) %>%
  summarise(n_cells = n(), .groups="drop") %>% as.data.frame()
donor_covs$.key <- apply(donor_covs[,gbc,drop=FALSE], 1, function(r)
    paste(gsub("_","-", as.character(r)), collapse="_"))
cat("Expected pseudobulk samples:", nrow(donor_covs), "\n")

# aggregate
pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)
cat("AggregateExpression produced:", ncol(pseudo), "samples\n")
cat("\nFirst few pseudobulk sample names (the keys Seurat built):\n")
print(head(colnames(pseudo), 4))
cat("\nFirst few of our constructed keys:\n")
print(head(donor_covs$.key, 4))

# merge n_cells
m <- match(colnames(pseudo), donor_covs$.key)
if (any(is.na(m))) {
  cat("\n⚠ direct key match missed", sum(is.na(m)), "— trying rebuild from pseudo meta\n")
  pm <- pseudo@meta.data[, gbc, drop=FALSE]
  pkey <- apply(pm, 1, function(r) paste(gsub("_","-",as.character(r)), collapse="_"))
  m <- match(pkey, donor_covs$.key)
}
pseudo$n_cells <- donor_covs$n_cells[m]
cat(sprintf("\nmatched n_cells: %d / %d samples\n", sum(!is.na(pseudo$n_cells)), ncol(pseudo)))
cat("n_cells summary:\n"); print(summary(pseudo$n_cells))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 4 — min-cell filter + re-pair (drop thin samples & unpaired donors)
# ════════════════════════════════════════════════════════════════════════════
MIN_CELLS <- 10

nc <- pseudo$n_cells
cat("Before filter:", ncol(pseudo), "samples\n")
pseudo <- pseudo[, !is.na(nc) & nc >= MIN_CELLS]
cat("After >=", MIN_CELLS, "cells:", ncol(pseudo), "samples\n")

pseudo$pop <- factor(pseudo$pop, levels=c("REF","TEST"))
pd <- pseudo$Donor; pp <- pseudo$pop
tab <- table(pd, pp)
paired <- rownames(tab)[tab[,"TEST"]>0 & tab[,"REF"]>0]
pseudo <- pseudo[, pd %in% paired]
cat("After re-pairing (donor has both pops):", ncol(pseudo), "samples |", length(paired), "donors\n\n")

cat("Final pop x grp2 (sample counts):\n"); print(table(pseudo$pop, pseudo$grp2))
cat("\nFinal Sex x pop:\n"); print(table(pseudo$Sex, pseudo$pop))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 5 — DESIGN MATRIX (~ pop + grp2 + Sex, WITH intercept) + verify contrast
#   popTEST coefficient = TEST - REF (because intercept present + REF is baseline)
# ════════════════════════════════════════════════════════════════════════════
meta <- pseudo@meta.data
meta$pop  <- factor(meta$pop,  levels=c("REF","TEST"))   # REF = baseline
meta$grp2 <- factor(meta$grp2)
meta$Sex  <- factor(meta$Sex)

# intercept design — NOT 0+ ; popTEST is already the difference
design <- model.matrix(~ pop + grp2 + Sex, data=meta)

cat("Design columns:\n"); print(colnames(design))
cat("\nDesign rank:", qr(design)$rank, "of", ncol(design), "columns",
    ifelse(qr(design)$rank==ncol(design), "(full rank ✓)", "(RANK DEFICIENT ✗)"), "\n")
cat("\nColumn sums (how many samples load on each):\n"); print(colSums(design))
cat("\nHead of design:\n"); print(head(design, 4))
cat("\n>>> The coefficient we will test is 'popTEST'.\n")
cat(">>> With the intercept present and REF as baseline,\n")
cat(">>> popTEST = (mean expression in TEST) - (mean expression in REF) = the contrast we want.\n")
cat(">>> This is NOT the 0+ trap (which tested absolute expression). \n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 6 — voom + donor-blocked limma + eBayes · test coef = popTEST (TEST - REF)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(edgeR); library(limma) })

counts <- as.matrix(GetAssayData(pseudo, layer="counts"))
cat("Count matrix:", nrow(counts), "genes x", ncol(counts), "samples\n")

dge  <- DGEList(counts=counts, samples=meta, group=meta$pop)
keep <- filterByExpr(dge, group=meta$pop)
dge  <- dge[keep, , keep.lib.sizes=FALSE]
dge  <- calcNormFactors(dge, method="TMM")
cat("Genes after filterByExpr:", nrow(dge), "\n")

v <- voom(dge, design, plot=FALSE)
corfit <- duplicateCorrelation(v, design, block=meta$Donor)
cat(sprintf("Donor consensus correlation: %.3f\n", corfit$consensus.correlation))

fit <- lmFit(v, design, block=meta$Donor, correlation=corfit$consensus.correlation)
fit <- eBayes(fit)

# ── test the popTEST coefficient (= TEST - REF) ──
res <- topTable(fit, coef="popTEST", number=Inf, sort.by="P")
res$gene <- rownames(res)
res$direction <- ifelse(res$adj.P.Val<0.05 & res$logFC>0, "Up",
                 ifelse(res$adj.P.Val<0.05 & res$logFC<0, "Down","NS"))

cat(sprintf("\n>>> RESULTS: %s vs %s\n", TEST, REF))
cat(sprintf("  tested: %d genes\n", nrow(res)))
cat(sprintf("  Up in %s (TEST):  %d\n", TEST, sum(res$direction=="Up")))
cat(sprintf("  Down (up in %s):  %d\n", REF, sum(res$direction=="Down")))
cat(sprintf("  NS:               %d\n", sum(res$direction=="NS")))
cat(sprintf("  logFC range: [%.2f, %.2f]\n", min(res$logFC), max(res$logFC)))
cat("\n  top 8 up in TEST:\n"); print(head(res[res$direction=="Up", c("gene","logFC","adj.P.Val")], 8))
cat("\n  top 8 up in REF:\n"); print(head(res[res$direction=="Down", c("gene","logFC","adj.P.Val")], 8))

---
## 04 · Four-contrast loop — validated flow

**Why.** One loop, four contrasts, identical machinery — so a difference between contrasts is biology and not procedure.

**Test.** limma-voom, `duplicateCorrelation` blocking on donor, which handles donors contributing to both arms.

**Formula.** `~ pop + grp2 + Sex`, testing `popTEST` (TEST − REF).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 7 — 4-CONTRAST loop · validated flow (~ pop + grp2 + Sex, coef=popTEST)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr); library(edgeR); library(limma) })
DONOR_COL<-"Donor"; SEX_COL<-"Sex"; MIN_CELLS<-10
BASE_OUT <- file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")
dir.create(BASE_OUT, recursive=TRUE, showWarnings=FALSE)

CONTRASTS <- list(
  DAMaxis_SnCpos = c("SnCpos_DAMpos","SnCpos_DAMneg"),
  DAMaxis_SnCneg = c("SnCneg_DAMpos","SnCneg_DAMneg"),
  SenAxis_DAMpos = c("SnCpos_DAMpos","SnCneg_DAMpos"),
  SenAxis_DAMneg = c("SnCpos_DAMneg","SnCneg_DAMneg"))

run_de <- function(tag, test, ref) {
  cat("\n", strrep("=",60), "\n", tag, "  (", test, " vs ", ref, ")\n", strrep("=",60), "\n", sep="")
  OBJ <- obj_ct[, obj_ct$quad %in% c(test,ref) & !is.na(obj_ct$grp2)]
  OBJ$pop <- factor(ifelse(OBJ$quad==test,"TEST","REF"), levels=c("REF","TEST"))

  gbc <- c(DONOR_COL,"grp2","pop",SEX_COL); gbc <- gbc[gbc %in% colnames(OBJ@meta.data)]
  donor_covs <- OBJ@meta.data %>% group_by(across(all_of(gbc))) %>%
    summarise(n_cells=n(), .groups="drop") %>% as.data.frame()
  donor_covs$.key <- apply(donor_covs[,gbc,drop=FALSE],1,function(r) paste(gsub("_","-",as.character(r)),collapse="_"))

  pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)
  m <- match(colnames(pseudo), donor_covs$.key)
  if (any(is.na(m))) { pm<-pseudo@meta.data[,gbc,drop=FALSE]
    m<-match(apply(pm,1,function(r) paste(gsub("_","-",as.character(r)),collapse="_")), donor_covs$.key) }
  pseudo$n_cells <- donor_covs$n_cells[m]

  # min-cell + re-pair
  pseudo <- pseudo[, !is.na(pseudo$n_cells) & pseudo$n_cells >= MIN_CELLS]
  pseudo$pop <- factor(pseudo$pop, levels=c("REF","TEST"))
  tab <- table(pseudo$Donor, pseudo$pop)
  paired <- rownames(tab)[tab[,"TEST"]>0 & tab[,"REF"]>0]
  pseudo <- pseudo[, pseudo$Donor %in% paired]
  cat(sprintf("  paired donors: %d | samples: %d\n", length(paired), ncol(pseudo)))

  meta <- pseudo@meta.data
  meta$pop<-factor(meta$pop,levels=c("REF","TEST")); meta$grp2<-factor(meta$grp2); meta$Sex<-factor(meta$Sex)

  # design ~ pop + grp2 + Sex (intercept); drop grp2/Sex if single-level
  terms <- "pop"
  for (cv in c("grp2", SEX_COL)) if (nlevels(meta[[cv]])>1) terms<-c(terms,cv) else cat(sprintf("  (drop %s: single level)\n",cv))
  design <- model.matrix(as.formula(paste("~", paste(terms,collapse=" + "))), data=meta)
  stopifnot(qr(design)$rank == ncol(design))     # guard: must be full rank
  stopifnot("popTEST" %in% colnames(design))     # guard: popTEST must be a single difference column

  counts <- as.matrix(GetAssayData(pseudo, layer="counts"))
  dge <- DGEList(counts=counts, samples=meta, group=meta$pop)
  keep <- filterByExpr(dge, group=meta$pop); dge <- dge[keep,,keep.lib.sizes=FALSE]
  dge <- calcNormFactors(dge, method="TMM")
  v <- voom(dge, design, plot=FALSE)
  corfit <- duplicateCorrelation(v, design, block=meta$Donor)
  fit <- lmFit(v, design, block=meta$Donor, correlation=corfit$consensus.correlation)
  fit <- eBayes(fit)

  res <- topTable(fit, coef="popTEST", number=Inf, sort.by="P"); res$gene <- rownames(res)
  res$direction <- ifelse(res$adj.P.Val<0.05 & res$logFC>0,"Up",
                   ifelse(res$adj.P.Val<0.05 & res$logFC<0,"Down","NS"))
  cat(sprintf("  corr %.3f | genes %d | DEGs: %d up %s / %d up %s\n",
      corfit$consensus.correlation, nrow(res),
      sum(res$direction=="Up"), test, sum(res$direction=="Down"), ref))
  write.csv(res, file.path(BASE_OUT, sprintf("pseudobulk_DE_%s_microglia.csv", tag)), row.names=FALSE)
  res
}

all_res <- list()
for (tag in names(CONTRASTS)) all_res[[tag]] <- run_de(tag, CONTRASTS[[tag]][1], CONTRASTS[[tag]][2])

cat("\n", strrep("=",60), "\nSUMMARY — ~pop+grp2+Sex (intercept, coef=popTEST)\n", strrep("=",60), "\n", sep="")
for (tag in names(all_res)) { r<-all_res[[tag]]
  cat(sprintf("  %-16s %4d up / %4d down (of %d)\n", tag, sum(r$direction=="Up"), sum(r$direction=="Down"), nrow(r))) }
cat("\n✓ all four done — all_res in scope, CSVs written.\n")

---
## 05 · Volcano panel

**Why.** Four volcanoes on shared axes. Shared limits matter — otherwise a weak contrast is rescaled into looking strong.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2x2 VOLCANO PANEL — four quadrant contrasts · red=up(left)/blue=up(right)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel); library(patchwork) })
FDR_CUT<-0.05; LFC_CUT<-0.25; N_LABEL<-10; UP_COL<-"#C0392B"; DN_COL<-"#2471A3"
FIG_DIR<-file.path(SCRATCH, "brain/module_06_dge/figures"); dir.create(FIG_DIR,recursive=TRUE,showWarnings=FALSE)
if (!exists("save_figure")) save_figure<-function(p,n,width=6,height=5,dpi=300){for(f in c("png","pdf","svg")) ggsave(file.path(FIG_DIR,paste0(n,".",f)),p,width=width,height=height,dpi=dpi,bg="white",device=if(f=="pdf")cairo_pdf else f); cat(sprintf("  saved %s.{png,pdf,svg}\n",n))}

PANELS <- list(
  DAMaxis_SnCpos = list(t="DAM axis | senescent\nSnC+DAM+ vs SnC+DAM-",     up="SnC+DAM+", dn="SnC+DAM-"),
  DAMaxis_SnCneg = list(t="DAM axis | non-senescent\nSnC-DAM+ vs SnC-DAM-", up="SnC-DAM+", dn="SnC-DAM-"),
  SenAxis_DAMpos = list(t="Sen axis | DAM+\nSnC+DAM+ vs SnC-DAM+",          up="SnC+DAM+", dn="SnC-DAM+"),
  SenAxis_DAMneg = list(t="Sen axis | DAM-\nSnC+DAM- vs SnC-DAM-",          up="SnC+DAM-", dn="SnC-DAM-"))

one_volcano <- function(res, cfg) {
  d <- res %>% mutate(neglog10=-log10(adj.P.Val),
    cls=case_when(adj.P.Val<FDR_CUT & logFC>= LFC_CUT~"UP",
                  adj.P.Val<FDR_CUT & logFC<=-LFC_CUT~"DN", TRUE~"NS"))
  YCAP<-quantile(d$neglog10[is.finite(d$neglog10)],0.999); d$y<-pmin(d$neglog10,YCAP); d$capped<-d$neglog10>YCAP
  xlim<-max(abs(d$logFC))*1.05
  lab<-bind_rows(d%>%filter(cls=="UP")%>%arrange(adj.P.Val)%>%head(N_LABEL),
                 d%>%filter(cls=="DN")%>%arrange(adj.P.Val)%>%head(N_LABEL))
  nU<-sum(d$cls=="UP"); nD<-sum(d$cls=="DN")
  ggplot(d,aes(logFC,y))+
    geom_vline(xintercept=c(-LFC_CUT,LFC_CUT),linetype="dashed",colour="grey75",linewidth=0.25)+
    geom_hline(yintercept=-log10(FDR_CUT),linetype="dashed",colour="grey75",linewidth=0.25)+
    geom_point(data=subset(d,cls=="NS"),colour="grey80",size=0.5,alpha=0.3,shape=16)+
    geom_point(data=subset(d,cls!="NS"),aes(colour=cls),size=0.9,alpha=0.8,shape=16)+
    geom_point(data=subset(d,capped&cls!="NS"),aes(colour=cls),y=YCAP,shape=2,size=1.2,stroke=0.4)+
    geom_text_repel(data=lab,aes(label=gene),colour="black",size=2.2,fontface="italic",
                    segment.size=0.2,segment.color="grey60",min.segment.length=0,box.padding=0.28,max.overlaps=Inf)+
    annotate("text",x=xlim*0.95,y=0,hjust=1,vjust=0,size=2.4,colour=UP_COL,label=sprintf("%d up %s",nU,cfg$up))+
    annotate("text",x=-xlim*0.95,y=0,hjust=0,vjust=0,size=2.4,colour=DN_COL,label=sprintf("%d up %s",nD,cfg$dn))+
    scale_colour_manual(values=c(UP=UP_COL,DN=DN_COL,NS="grey80"),guide="none")+
    scale_x_continuous(limits=c(-xlim,xlim))+
    labs(x=expression(log[2]~FC),y=expression(-log[10]~FDR),title=cfg$t)+
    theme_classic(base_size=8)+
    theme(plot.title=element_text(face="bold",size=8,hjust=0,lineheight=1.05),
          axis.text=element_text(size=7,colour="black"),
          panel.border=element_rect(colour="black",fill=NA,linewidth=0.5),
          axis.line=element_blank(),axis.ticks=element_line(linewidth=0.35),plot.margin=margin(4,6,4,4))
}
plots<-lapply(names(PANELS),function(nm) one_volcano(all_res[[nm]],PANELS[[nm]]))
combined<-(plots[[1]]|plots[[2]])/(plots[[3]]|plots[[4]])+
  plot_annotation(title="Senescence x DAM: axis-resolved transcriptional programs (microglia)",
                  subtitle="Red = up in left term  ·  Blue = up in right term",
                  theme=theme(plot.title=element_text(face="bold",size=11),plot.subtitle=element_text(size=8,color="grey40")))
options(repr.plot.width=10,repr.plot.height=9); print(combined)
save_figure(combined,"volcano_2x2_quadrant_axes_microglia",width=10,height=9)

---
## 06 · DEG counts

**Why.** The four contrasts as diverging bars, up to the right and down to the left. An asymmetry between the activation contrasts and the senescence contrasts becomes visible here.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DEG COUNT — diverging bars (compact): up→right(red), down→left(blue)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(tidyr); library(stringr) })
FDR_CUT<-0.05; LFC_CUT<-0.25; UP_COL<-"#C0392B"; DN_COL<-"#2471A3"
FIG_DIR<-file.path(SCRATCH, "brain/module_06_dge/figures"); dir.create(FIG_DIR,recursive=TRUE,showWarnings=FALSE)
if (!exists("save_figure")) save_figure<-function(p,n,width=6,height=5,dpi=300){for(f in c("png","pdf","svg")) ggsave(file.path(FIG_DIR,paste0(n,".",f)),p,width=width,height=height,dpi=dpi,bg="white",device=if(f=="pdf")cairo_pdf else f); cat(sprintf("  saved %s.{png,pdf,svg}\n",n))}

META<-tibble::tribble(~key,~axis,~label,
  "DAMaxis_SnCpos","DAM axis","SnC+DAM+ vs SnC+DAM-",
  "DAMaxis_SnCneg","DAM axis","SnC-DAM+ vs SnC-DAM-",
  "SenAxis_DAMpos","Sen axis","SnC+DAM+ vs SnC-DAM+",
  "SenAxis_DAMneg","Sen axis","SnC+DAM- vs SnC-DAM-")
cnt<-lapply(META$key,function(k){r<-all_res[[k]]
  data.frame(key=k,Up=sum(r$adj.P.Val<FDR_CUT&r$logFC>=LFC_CUT),Down=-sum(r$adj.P.Val<FDR_CUT&r$logFC<=-LFC_CUT))})%>%
  bind_rows()%>%left_join(META,by="key")%>%pivot_longer(c(Up,Down),names_to="dir",values_to="n")
cnt$label<-factor(str_wrap(cnt$label,12),levels=str_wrap(rev(META$label),12))
cnt$dir<-factor(cnt$dir,levels=c("Up","Down"))
xlo<-min(cnt$n)*1.12; xhi<-max(cnt$n)*1.12
p<-ggplot(cnt,aes(x=n,y=label,fill=dir))+
  geom_vline(xintercept=0,colour="grey40",linewidth=0.4)+
  geom_col(width=0.6,colour="black",linewidth=0.25)+
  geom_text(aes(label=abs(n),hjust=ifelse(n>=0,-0.25,1.25)),size=2.9,fontface="bold")+
  facet_grid(axis~.,scales="free_y",space="free_y")+
  scale_fill_manual(values=c(Up=UP_COL,Down=DN_COL),name=NULL,labels=c("up in left term","up in right term"))+
  scale_x_continuous(limits=c(xlo,xhi),labels=function(x) abs(x))+
  labs(x="# of DEGs",y=NULL,title="DEG counts: senescence x DAM axis contrasts (microglia)")+
  theme_classic(base_size=10)+
  theme(plot.title=element_text(face="bold",size=10.5),axis.text.y=element_text(size=8,lineheight=0.9),
        strip.background=element_blank(),strip.text=element_text(face="bold",size=9.5),
        panel.border=element_rect(colour="black",fill=NA,linewidth=0.5),axis.line=element_blank(),
        legend.position="top",plot.margin=margin(4,6,4,4))
options(repr.plot.width=6.5,repr.plot.height=4); print(p)
save_figure(p,"DEG_counts_diverging_2x2_axes_microglia",width=6.5,height=4)

---
## 07 · Venns — consistency and distinctness

**Why.** Two questions. *Within* an axis, do the SnC+ and SnC− contrasts recover the same genes — is the activation effect stable across senescence strata? *Between* axes, are the senescence and activation gene sets distinct?

Large within-axis overlap plus small between-axis overlap is the separability result at gene level — module 09's claim, tested with genes instead of scores.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VENN — within-axis (DAM, Sen consistency) + between-axis distinctness
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr); library(ggplot2); library(patchwork) })
FDR_CUT<-0.05; LFC_CUT<-0.25; FILL_A<-"#6BAED6"; FILL_B<-"#FB9A99"
FIG_DIR<-file.path(SCRATCH, "brain/module_06_dge/figures"); dir.create(FIG_DIR,recursive=TRUE,showWarnings=FALSE)
if (!exists("save_figure")) save_figure<-function(p,n,width=6,height=5,dpi=300){for(f in c("png","pdf","svg")) ggsave(file.path(FIG_DIR,paste0(n,".",f)),p,width=width,height=height,dpi=dpi,bg="white",device=if(f=="pdf")cairo_pdf else f); cat(sprintf("  saved %s.{png,pdf,svg}\n",n))}

degs<-function(key){r<-all_res[[key]]; r$gene[r$adj.P.Val<FDR_CUT & abs(r$logFC)>=LFC_CUT]}
circle<-function(cx,cy,r,n=200){t<-seq(0,2*pi,length.out=n);data.frame(x=cx+r*cos(t),y=cy+r*sin(t))}
report<-function(a,b,na,nb){ov<-length(intersect(a,b));jac<-ov/max(1,length(union(a,b)))
  cat(sprintf("  %-15s %4d | shared %4d | %-15s %4d  (Jaccard %.2f)\n",na,length(setdiff(a,b)),ov,nb,length(setdiff(b,a)),jac))}
venn2<-function(a,b,labA,labB,title){
  oa<-length(setdiff(a,b));ob<-length(setdiff(b,a));ov<-length(intersect(a,b));jac<-ov/max(1,length(union(a,b)))
  R<-0.85;cxL<--0.5;cxR<-0.5
  cL<-circle(cxL,0,R);cL$set<-"A";cR<-circle(cxR,0,R);cR$set<-"B"
  ggplot()+geom_polygon(data=rbind(cL,cR),aes(x,y,group=set,fill=set),colour="black",linewidth=0.5,alpha=0.6)+
    scale_fill_manual(values=c(A=FILL_A,B=FILL_B),guide="none")+
    annotate("text",x=cxL-0.40,y=0,label=oa,fontface="bold",size=4)+
    annotate("text",x=0,y=0,label=ov,fontface="bold",size=4)+
    annotate("text",x=cxR+0.40,y=0,label=ob,fontface="bold",size=4)+
    annotate("text",x=-2.05,y=0,label=labA,fontface="bold",size=2.8,lineheight=0.9,hjust=0,colour="#2C5985")+
    annotate("text",x=2.05,y=0,label=labB,fontface="bold",size=2.8,lineheight=0.9,hjust=1,colour="#9E3B3B")+
    annotate("text",x=0,y=-R-0.35,label=sprintf("Jaccard = %.2f",jac),size=3.2,colour="grey30")+
    coord_fixed(xlim=c(-2.1,2.1),ylim=c(-1.5,1.2),clip="off")+labs(title=title)+theme_void()+
    theme(plot.title=element_text(face="bold",size=10.5,hjust=0.5,lineheight=0.95),plot.margin=margin(6,4,6,4))
}
cat("=== Jaccard overlaps (pooled DEGs) ===\n")
cat("WITHIN DAM:\n"); report(degs("DAMaxis_SnCpos"),degs("DAMaxis_SnCneg"),"DAM|SnC+","DAM|SnC-")
cat("WITHIN Sen:\n"); report(degs("SenAxis_DAMpos"),degs("SenAxis_DAMneg"),"Sen|DAM+","Sen|DAM-")
cat("BETWEEN:\n");    report(degs("DAMaxis_SnCpos"),degs("SenAxis_DAMpos"),"DAM axis","Sen axis")
v1<-venn2(degs("DAMaxis_SnCpos"),degs("DAMaxis_SnCneg"),
          "DAM axis\nin SnC+\n(SnC+DAM+ vs\nSnC+DAM-)","DAM axis\nin SnC-\n(SnC-DAM+ vs\nSnC-DAM-)",
          "DAM program:\nsenescent vs non-senescent")
v2<-venn2(degs("SenAxis_DAMpos"),degs("SenAxis_DAMneg"),
          "Sen axis\nin DAM+\n(SnC+DAM+ vs\nSnC-DAM+)","Sen axis\nin DAM-\n(SnC+DAM- vs\nSnC-DAM-)",
          "Sen program:\nDAM+ vs DAM-")
v3<-venn2(degs("DAMaxis_SnCpos"),degs("SenAxis_DAMpos"),
          "DAM axis\n(SnC+DAM+ vs\nSnC+DAM-)","Sen axis\n(SnC+DAM+ vs\nSnC-DAM+)","Between axes:\nDAM vs Sen")
panel<-(v1|v2|v3)+plot_annotation(title="Within-axis consistency vs between-axis distinctness (microglia DEGs)",
                  theme=theme(plot.title=element_text(face="bold",size=12,hjust=0)))
options(repr.plot.width=13,repr.plot.height=4.5); print(panel)
save_figure(panel,"venn_axis_overlaps_pooled",width=13,height=4.5)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VENN (pure ggplot) — labels placed OUTSIDE each circle, no overlap
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(dplyr); library(ggplot2); library(patchwork) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "SnC"; X_TAG <- "IRM"          # must match the DE run that filled all_res
Y_NAME <- "senescence"                  # spelled-out axis names for titles/labels
X_NAME <- "IRM"
FDR_CUT <- 0.05; LFC_CUT <- 0.25
FILL_A <- "#6BAED6"; FILL_B <- "#FB9A99"
FILE_TAG <- "SnC_IRM"
FIG_DIR <- file.path(SCRATCH, "brain/module_06_dge/figures")
# ════════════════════════════════════════════════════════════════════════════
dir.create(FIG_DIR, recursive=TRUE, showWarnings=FALSE)
if (!exists("save_figure")) {
  save_figure <- function(plot,name,width=6,height=5,dpi=300){
    for (fmt in c("png","pdf","svg"))
      ggsave(file.path(FIG_DIR,paste0(name,".",fmt)), plot, width=width, height=height,
             dpi=dpi, bg="white", device=if(fmt=="pdf") cairo_pdf else fmt)
    cat(sprintf("  saved: %s.{png,pdf,svg}\n", name)) }
}
# contrast keys (match the DE cell tags)
K_XpY <- sprintf("%saxis_%spos", X_TAG, Y_TAG)   # IRM axis | SnC+   clean
K_XnY <- sprintf("%saxis_%sneg", X_TAG, Y_TAG)   # IRM axis | SnC-   clean
K_YpX <- sprintf("%saxis_%spos", Y_TAG, X_TAG)   # Sen axis | IRM+   circular
K_YnX <- sprintf("%saxis_%sneg", Y_TAG, X_TAG)   # Sen axis | IRM-   circular

degs <- function(key, dir=c("all","up","down")){
  dir<-match.arg(dir); r<-all_res[[key]]
  if (is.null(r)) return(character(0))
  if(dir=="up")   return(r$gene[r$adj.P.Val<FDR_CUT & r$logFC>= LFC_CUT])
  if(dir=="down") return(r$gene[r$adj.P.Val<FDR_CUT & r$logFC<=-LFC_CUT])
  r$gene[r$adj.P.Val<FDR_CUT & abs(r$logFC)>=LFC_CUT]
}
circle <- function(cx,cy,r,n=200){ t<-seq(0,2*pi,length.out=n); data.frame(x=cx+r*cos(t), y=cy+r*sin(t)) }
venn2 <- function(a, b, labA, labB, title){
  only_a<-length(setdiff(a,b)); only_b<-length(setdiff(b,a)); ov<-length(intersect(a,b))
  jac <- ov / max(1, length(union(a,b)))
  R<-0.85; cxL<--0.5; cxR<-0.5
  cL<-circle(cxL,0,R); cL$set<-"A"; cR<-circle(cxR,0,R); cR$set<-"B"
  ggplot() +
    geom_polygon(data=rbind(cL,cR), aes(x,y,group=set,fill=set), colour="black", linewidth=0.5, alpha=0.6) +
    scale_fill_manual(values=c(A=FILL_A,B=FILL_B), guide="none") +
    annotate("text", x=cxL-0.40, y=0, label=only_a, fontface="bold", size=4) +
    annotate("text", x=0,        y=0, label=ov,     fontface="bold", size=4) +
    annotate("text", x=cxR+0.40, y=0, label=only_b, fontface="bold", size=4) +
    annotate("text", x=-2.05, y=0, label=labA, fontface="bold", size=2.8, lineheight=0.9, hjust=0, colour="#2C5985") +
    annotate("text", x= 2.05, y=0, label=labB, fontface="bold", size=2.8, lineheight=0.9, hjust=1, colour="#9E3B3B") +
    annotate("text", x=0, y=-R-0.35, label=sprintf("Jaccard = %.2f", jac), size=3.2, colour="grey30") +
    coord_fixed(xlim=c(-2.1,2.1), ylim=c(-1.5,1.2), clip="off") +
    labs(title=title) + theme_void() +
    theme(plot.title=element_text(face="bold", size=10.5, hjust=0.5, lineheight=0.95),
          plot.margin=margin(6,4,6,4))
}
yp<-paste0(Y_TAG,"+"); yn<-paste0(Y_TAG,"-"); xp<-paste0(X_TAG,"+"); xn<-paste0(X_TAG,"-")

# v1: IRM program consistency — SnC+ vs SnC-   (both CLEAN)
v1 <- venn2(degs(K_XpY), degs(K_XnY),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", X_NAME, yp, yp,xp, yp,xn),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", X_NAME, yn, yn,xp, yn,xn),
            sprintf("%s program:\nsenescent vs non-senescent", X_NAME))
# v2: senescence program consistency — IRM+ vs IRM-   (both CIRCULAR)
v2 <- venn2(degs(K_YpX), degs(K_YnX),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", Y_NAME, xp, yp,xp, yn,xp),
            sprintf("%s axis\nin %s\n(%s%s vs\n%s%s)", Y_NAME, xn, yp,xn, yn,xn),
            sprintf("%s program:\n%s vs %s", Y_NAME, xp, xn))
# v3: between axes — clean IRM axis vs circular senescence axis, in the SnC+IRM+ corner
v3 <- venn2(degs(K_XpY), degs(K_YpX),
            sprintf("%s axis\n(%s%s vs\n%s%s)", X_NAME, yp,xp, yp,xn),
            sprintf("%s axis \n(%s%s vs\n%s%s)", Y_NAME, yp,xp, yn,xp),
            sprintf("Between axes:\n%s vs %s", X_NAME, Y_NAME))

panel <- (v1 | v2 | v3) +
  plot_annotation(title=sprintf("Within-axis consistency vs between-axis distinctness (microglia DEGs, %s)", X_NAME),
                  theme=theme(plot.title=element_text(face="bold", size=12, hjust=0)))
options(repr.plot.width=13, repr.plot.height=4.5); print(panel)
save_figure(panel, sprintf("venn_axis_overlaps_pooled_%s", FILE_TAG), width=13, height=4.5)
cat("\n\u2713 done\n")

---
## 08 · Within-axis split among senescent cells

**Why.** `SenHi_Xhi` versus `SenHi_Xlo` — the activation contrast restricted to high-senescence cells, run on its own so the comparison is not diluted by the non-senescent arm.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PSEUDOBULK DE — SenHi_Xhi vs SenHi_Xlo  (among high-sen cells, split on X axis)
#   ref = SenHi_Xlo, so logFC = Xhi − Xlo   ·   X non-circular w.r.t. senescence
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr); library(edgeR); library(limma) })

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
OBJ_IN <- obj_ct
Y_COL  <- "senescence_score"; Y_TAG <- "Sen"   # held HIGH in both pops
X_COL  <- "Score_IRM";   X_TAG <- "IRM"   # the split axis (Hi vs Lo)
GROUPS <- c("Old_AD"="AD", "Old_Healthy_Control"="Control")
DONOR_COL<-"Donor"; SEX_COL<-"Sex"; AGE_COL<-"Age"; COHORT_COL<-"Cohort"
HAS_COHORT <- TRUE; MIN_CELLS <- 10
FDR_THRESHOLD <- 0.05
# ════════════════════════════════════════════════════════════════════════════
POP_HI <- sprintf("%sHi_%shi", Y_TAG, X_TAG)   # e.g. SenHi_DAMhi
POP_LO <- sprintf("%sHi_%slo", Y_TAG, X_TAG)   # e.g. SenHi_DAMlo
HI <- gsub("_","-",POP_HI); LO <- gsub("_","-",POP_LO)   # dash form after Aggregate
XHI <- sprintf("%shi", X_TAG); XLO <- sprintf("%slo", X_TAG)

# ── 1a: define the two SenHi quadrants, subset ──────────────────────────────
OBJ <- OBJ_IN; md <- OBJ@meta.data
stopifnot(Y_COL %in% colnames(md), X_COL %in% colnames(md))
y_z <- as.numeric(scale(md[[Y_COL]])); x_z <- as.numeric(scale(md[[X_COL]]))
OBJ$popX <- ifelse(y_z>0 & x_z>0,  POP_HI,
            ifelse(y_z>0 & x_z<=0, POP_LO, NA))
OBJ$grp2 <- unname(GROUPS[as.character(md$Study_Group)])
OBJ <- OBJ[, !is.na(OBJ$popX) & !is.na(OBJ$grp2)]
cat("Cells per popX \u00d7 grp2:\n"); print(table(OBJ$popX, OBJ$grp2))

# ── 1b: covariates ───────────────────────────────────────────────────────────
if (!"nCount_RNA" %in% colnames(OBJ@meta.data)) OBJ$nCount_RNA <- colSums(GetAssayData(OBJ, slot="counts"))
gbc <- c(DONOR_COL,"grp2","popX",SEX_COL,COHORT_COL)
donor_covs <- OBJ@meta.data %>% group_by(across(all_of(gbc))) %>%
    summarise(Mean_Age=mean(as.numeric(.data[[AGE_COL]]),na.rm=TRUE),
              Mean_Log_Library_Depth=mean(log10(nCount_RNA+1),na.rm=TRUE),
              n_cells=n(), .groups="drop") %>% as.data.frame()

# ── 1c: aggregate ────────────────────────────────────────────────────────────
pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)
cat(sprintf("\nPseudobulk: %d samples\n", ncol(pseudo)))

# ── 1d: pair + covariates (dash form) ───────────────────────────────────────
pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tab <- table(pb_donor, pseudo[["popX",drop=TRUE]])
stopifnot(all(c(HI,LO) %in% colnames(tab)))
paired <- rownames(tab)[tab[,HI]>0 & tab[,LO]>0]
pseudo <- pseudo[, pb_donor %in% paired]
for (col in gbc) if (is.character(donor_covs[[col]])||is.factor(donor_covs[[col]]))
    donor_covs[[col]] <- gsub("_","-",as.character(donor_covs[[col]]))
pm <- pseudo@meta.data; pm$row_id <- rownames(pm); for (col in gbc) pm[[col]] <- as.character(pm[[col]])
mgd <- merge(pm, donor_covs, by=gbc, all.x=TRUE, sort=FALSE); mgd <- mgd[match(pm$row_id, mgd$row_id),]
pseudo[["Mean_Age"]]<-mgd$Mean_Age; pseudo[["Mean_Log_Library_Depth"]]<-mgd$Mean_Log_Library_Depth
pseudo[["n_cells"]]<-mgd$n_cells
cat(sprintf("paired donors: %d | samples: %d\n", length(paired), ncol(pseudo)))

# ── 1e: cell filter + re-pair ────────────────────────────────────────────────
pseudo <- pseudo[, pseudo[["n_cells",drop=TRUE]] >= MIN_CELLS]
pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tab <- table(pb_donor, pseudo[["popX",drop=TRUE]])
paired <- rownames(tab)[tab[,HI]>0 & tab[,LO]>0]; pseudo <- pseudo[, pb_donor %in% paired]
cat(sprintf("after n_cells>=%d + re-pair: %d donors, %d samples\n", MIN_CELLS, length(paired), ncol(pseudo)))
cat("Final samples per popX \u00d7 grp2:\n"); print(table(pseudo[["popX",drop=TRUE]], pseudo[["grp2",drop=TRUE]]))

# ── 1f: DGEList + filter + TMM ───────────────────────────────────────────────
counts_mat <- as.matrix(GetAssayData(pseudo, layer="counts")); meta_df <- pseudo@meta.data
meta_df[[SEX_COL]]<-factor(meta_df[[SEX_COL]]); meta_df[[DONOR_COL]]<-factor(meta_df[[DONOR_COL]])
if (HAS_COHORT){ meta_df[[COHORT_COL]]<-factor(meta_df[[COHORT_COL]]); levels(meta_df[[COHORT_COL]])<-make.names(levels(meta_df[[COHORT_COL]])) }
meta_df$pop  <- factor(ifelse(meta_df$popX==HI, XHI, XLO), levels=c(XLO, XHI))  # ref = Xlo
meta_df$grp2 <- factor(meta_df$grp2)
meta_df$Mean_Log_Library_Depth_scaled <- scale(meta_df$Mean_Log_Library_Depth)[,1]
dge <- DGEList(counts=counts_mat, samples=meta_df, group=meta_df$pop)
keep <- filterByExpr(dge, group=meta_df$pop); dge <- dge[keep,,keep.lib.sizes=FALSE]
dge <- calcNormFactors(dge, method="TMM")
cat(sprintf("\ngenes retained: %d / %d | pop ref=%s (logFC = %s - %s)\n",
            nrow(dge), nrow(counts_mat), XLO, XHI, XLO))

# ── 1g: voom + donor block + contrast ────────────────────────────────────────
design_str <- paste0("~0 + pop + grp2 + ", SEX_COL, " + Mean_Log_Library_Depth_scaled",
                     if (HAS_COHORT) paste0(" + ", COHORT_COL) else "")
design <- model.matrix(as.formula(design_str), data=meta_df)
colnames(design) <- make.names(colnames(design))
cat(sprintf("design rank: %d/%d\n", qr(design)$rank, ncol(design)))
stopifnot(qr(design)$rank == ncol(design))
v <- voom(dge, design, plot=FALSE)
corfit <- duplicateCorrelation(v, design, block=meta_df[[DONOR_COL]])
cat(sprintf("donor consensus correlation: %.3f\n", corfit$consensus.correlation))
fit <- lmFit(v, design, block=meta_df[[DONOR_COL]], correlation=corfit$consensus.correlation)
hi_col <- make.names(paste0("pop", XHI)); lo_col <- make.names(paste0("pop", XLO))
stopifnot(hi_col %in% colnames(design), lo_col %in% colnames(design))
fit <- contrasts.fit(fit, makeContrasts(contrasts=paste(hi_col,"-",lo_col), levels=design))
fit <- eBayes(fit)
res <- topTable(fit, coef=1, number=Inf, sort.by="P")
res$gene <- rownames(res)
res$direction <- ifelse(res$adj.P.Val<FDR_THRESHOLD & res$logFC>0,"Up",
                 ifelse(res$adj.P.Val<FDR_THRESHOLD & res$logFC<0,"Down","NS"))
cat(sprintf("\nDEGs FDR<%.2f: %d up, %d down (of %d tested)\n",
            FDR_THRESHOLD, sum(res$direction=="Up"), sum(res$direction=="Down"), nrow(res)))
cat("\nTop 15:\n"); print(head(res[,c("gene","logFC","P.Value","adj.P.Val","direction")], 15))
save_table(res, sprintf("pseudobulk_DE_%sHi%shi_vs_%sHi%slo_microglia", Y_TAG, X_TAG, Y_TAG, X_TAG))
cat("\n\u2713 DE done \u2014 res in scope.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VOLCANO — SenHi_Xhi vs SenHi_Xlo, panel-colored + top labels
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel) })
if (!exists("res")) stop("\u2717 run the DE first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "Sen"; X_TAG <- "IRM"          # must match the DE that produced `res`
X_PANEL <- "IRM"                   # MICROGLIA_STATE_PANELS entry for the split axis
FDR_THRESHOLD <- 0.05
LAB_LFC <- 1.3                          # also label any sig gene with |logFC| beyond this
# highlight panels (name → color); first-listed wins ties via the override order below
HL <- list(SASP     = list(genes = unique(toupper(gene_lists$SASP)),                 col = "#E67E22"),
           DDR      = list(genes = unique(toupper(gene_lists$DDR)),                  col = "#2E86C1"),
           X        = list(genes = unique(toupper(MICROGLIA_STATE_PANELS[[X_PANEL]])), col = "#C0392B"))
names(HL)[names(HL)=="X"] <- X_PANEL    # rename so legend/labels read the real panel
# ════════════════════════════════════════════════════════════════════════════
XHI <- sprintf("%shi", X_TAG); XLO <- sprintf("%slo", X_TAG)

v <- res; v$G <- toupper(v$gene)
v$neglog10fdr <- -log10(pmax(v$adj.P.Val, 1e-300))

# class assignment — apply in list order, so LAST assigned wins (X panel overrides SASP/DDR)
v$cls <- "other"
for (pn in names(HL)) v$cls[v$G %in% HL[[pn]]$genes] <- pn
v$sig <- v$adj.P.Val < FDR_THRESHOLD
hl_names <- names(HL)
v$cls_plot <- ifelse(!v$sig, "ns", v$cls)
v$cls_plot <- factor(v$cls_plot, levels = c("ns","other", hl_names))

COL <- c(ns="grey85", other="grey55", setNames(sapply(HL, `[[`, "col"), names(HL)))

lab <- v %>% filter(sig) %>%
    group_by(cls) %>% slice_max(abs(logFC), n=6) %>% ungroup() %>%
    filter(cls %in% hl_names | abs(logFC) > LAB_LFC)

p <- ggplot(v, aes(logFC, neglog10fdr)) +
    geom_vline(xintercept=0, color="grey80", linewidth=0.3) +
    geom_hline(yintercept=-log10(FDR_THRESHOLD), color="grey70", linetype="dashed", linewidth=0.3) +
    geom_point(data=subset(v, cls_plot %in% c("ns","other")),
               aes(color=cls_plot), size=0.5, alpha=0.4) +
    geom_point(data=subset(v, cls_plot %in% hl_names),
               aes(color=cls_plot), size=1.4, alpha=0.9) +
    ggrepel::geom_text_repel(data=lab, aes(label=gene, color=cls), size=2.6,
        max.overlaps=20, segment.size=0.2, segment.color="grey60", show.legend=FALSE) +
    scale_color_manual(values=COL, name=NULL,
        breaks=c(hl_names, "other"), labels=c(hl_names, "other")) +
    labs(title=sprintf("%sHi_%shi vs %sHi_%slo (%s axis among senescent cells)",
                       Y_TAG, X_TAG, Y_TAG, X_TAG, X_TAG),
         subtitle=sprintf("logFC = %s - %s | %d up, %d down (FDR<%.2f) | dashed = FDR %.2f",
                          XHI, XLO, sum(v$direction=="Up"), sum(v$direction=="Down"),
                          FDR_THRESHOLD, FDR_THRESHOLD),
         x=sprintf("log2 fold-change (%s - %s)", XHI, XLO), y="-log10 FDR") +
    theme_clean() +
    theme(plot.subtitle=element_text(size=7, color="grey45"), legend.position="right")
options(repr.plot.width=8, repr.plot.height=6); print(p)
save_figure(p, sprintf("volcano_%sHi%shi_vs_%sHi%slo_microglia", Y_TAG, X_TAG, Y_TAG, X_TAG), width=8, height=6)
cat("\n\u2713 volcano done.\n")

In [ ]:
# AXIS: 1 hardcoded 'Score_DAM_like' reference(s) -> X_COL,
#       resolved from AXIS in the config cell.
# ════════════════════════════════════════════════════════════════════════════
# SenHi-DAMhi vs SenHi-DAMlo (microglia) — pseudobulk DE (limma-voom) ONLY
#   ref = SenHi-DAMlo → logFC > 0 = up in DAM+ ; logFC < 0 = up in DAM− (sen-canonical)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(Seurat); library(dplyr); library(edgeR); library(limma) })

gmap <- c(Old_AD="AD", Old_Healthy_Control="Control")
DONOR_COL<-"Donor"; SEX_COL<-"Sex"; AGE_COL<-"Age"; COHORT_COL<-"Cohort"; MIN_CELLS<-10
OBJ <- obj_ct                              # ← must be the MICROGLIA object

# ── 1a: define the two SenHi quadrants, subset ──────────────────────────────
md <- OBJ@meta.data
sen_z <- as.numeric(scale(md$senescence_score)); dam_z <- as.numeric(scale(md[[X_COL]]))
OBJ$popX <- ifelse(sen_z>0 & dam_z>0,  "SenHi_DAMhi",
            ifelse(sen_z>0 & dam_z<=0, "SenHi_DAMlo", NA))
OBJ$grp2 <- unname(gmap[as.character(md$Study_Group)])
OBJ <- OBJ[, !is.na(OBJ$popX) & !is.na(OBJ$grp2)]
cat("Cells per popX × grp2:\n"); print(table(OBJ$popX, OBJ$grp2))

# ── 1b: covariates (donor-level means) ───────────────────────────────────────
if (!"nCount_RNA" %in% colnames(OBJ@meta.data))
  OBJ$nCount_RNA <- colSums(GetAssayData(OBJ, layer="counts"))
gbc <- c(DONOR_COL,"grp2","popX",SEX_COL,COHORT_COL)
donor_covs <- OBJ@meta.data %>% group_by(across(all_of(gbc))) %>%
  summarise(Mean_Age=mean(as.numeric(.data[[AGE_COL]]),na.rm=TRUE),
            Mean_Log_Library_Depth=mean(log10(nCount_RNA+1),na.rm=TRUE),
            n_cells=n(), .groups="drop") %>% as.data.frame()

# ── 1c: aggregate to pseudobulk ──────────────────────────────────────────────
pseudo <- AggregateExpression(OBJ, assays="RNA", return.seurat=TRUE, group.by=gbc)
cat(sprintf("\nPseudobulk: %d samples\n", ncol(pseudo)))

# ── 1d: pair donors (both DAM hi & lo) + attach covariates (dash form) ───────
HI<-"SenHi-DAMhi"; LO<-"SenHi-DAMlo"
pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tab <- table(pb_donor, pseudo[["popX",drop=TRUE]])
paired <- rownames(tab)[tab[,HI]>0 & tab[,LO]>0]
pseudo <- pseudo[, pb_donor %in% paired]
for (col in gbc) if (is.character(donor_covs[[col]])||is.factor(donor_covs[[col]]))
  donor_covs[[col]] <- gsub("_","-",as.character(donor_covs[[col]]))
pm <- pseudo@meta.data; pm$row_id <- rownames(pm); for (col in gbc) pm[[col]] <- as.character(pm[[col]])
mgm <- merge(pm, donor_covs, by=gbc, all.x=TRUE, sort=FALSE); mgm <- mgm[match(pm$row_id, mgm$row_id),]
pseudo[["Mean_Age"]]<-mgm$Mean_Age; pseudo[["Mean_Log_Library_Depth"]]<-mgm$Mean_Log_Library_Depth
pseudo[["n_cells"]]<-mgm$n_cells
cat(sprintf("paired donors: %d | samples: %d\n", length(paired), ncol(pseudo)))

# ── 1e: cell filter + re-pair ────────────────────────────────────────────────
pseudo <- pseudo[, pseudo[["n_cells",drop=TRUE]] >= MIN_CELLS]
pb_donor <- pseudo[[DONOR_COL,drop=TRUE]]; tab <- table(pb_donor, pseudo[["popX",drop=TRUE]])
paired <- rownames(tab)[tab[,HI]>0 & tab[,LO]>0]; pseudo <- pseudo[, pb_donor %in% paired]
cat(sprintf("after n_cells>=%d + re-pair: %d donors, %d samples\n", MIN_CELLS, length(paired), ncol(pseudo)))
cat("Final samples per popX × grp2:\n"); print(table(pseudo[["popX",drop=TRUE]], pseudo[["grp2",drop=TRUE]]))

# ── 1f: DGEList + filter + TMM ───────────────────────────────────────────────
counts_mat <- as.matrix(GetAssayData(pseudo, layer="counts")); meta_df <- pseudo@meta.data
meta_df[[SEX_COL]]<-factor(meta_df[[SEX_COL]]); meta_df[[DONOR_COL]]<-factor(meta_df[[DONOR_COL]])
meta_df[[COHORT_COL]]<-factor(meta_df[[COHORT_COL]]); levels(meta_df[[COHORT_COL]])<-make.names(levels(meta_df[[COHORT_COL]]))
meta_df$pop  <- factor(ifelse(meta_df$popX=="SenHi-DAMhi","DAMhi","DAMlo"), levels=c("DAMlo","DAMhi"))
meta_df$grp2 <- factor(meta_df$grp2)
meta_df$Mean_Log_Library_Depth_scaled <- scale(meta_df$Mean_Log_Library_Depth)[,1]
dge <- DGEList(counts=counts_mat, samples=meta_df, group=meta_df$pop)
keep <- filterByExpr(dge, group=meta_df$pop); dge <- dge[keep,,keep.lib.sizes=FALSE]
dge <- calcNormFactors(dge, method="TMM")
cat(sprintf("\ngenes retained: %d / %d | ref=DAMlo (logFC = DAMhi - DAMlo)\n", nrow(dge), nrow(counts_mat)))

# ── 1g: voom + donor block + contrast ────────────────────────────────────────
design <- model.matrix(~0 + pop + grp2 + Sex + Mean_Log_Library_Depth_scaled + Cohort, data=meta_df)
colnames(design) <- make.names(colnames(design))
cat(sprintf("design rank: %d/%d\n", qr(design)$rank, ncol(design)))
v <- voom(dge, design, plot=FALSE)
corfit <- duplicateCorrelation(v, design, block=meta_df[[DONOR_COL]])
cat(sprintf("donor consensus correlation: %.3f\n", corfit$consensus.correlation))
fit <- lmFit(v, design, block=meta_df[[DONOR_COL]], correlation=corfit$consensus.correlation)
fit <- contrasts.fit(fit, makeContrasts(popDAMhi - popDAMlo, levels=design))
fit <- eBayes(fit)
res <- topTable(fit, coef=1, number=Inf, sort.by="P"); res$gene <- rownames(res)
res$direction <- ifelse(res$adj.P.Val<0.05 & res$logFC>0,"Up_DAMpos",
                 ifelse(res$adj.P.Val<0.05 & res$logFC<0,"Up_DAMneg","NS"))
cat(sprintf("\nDEGs FDR<0.05: %d up in DAM+, %d up in DAM- (of %d tested)\n",
            sum(res$direction=="Up_DAMpos"), sum(res$direction=="Up_DAMneg"), nrow(res)))
cat("\nTop 15 by P:\n"); print(head(res[,c("gene","logFC","P.Value","adj.P.Val","direction")], 15))
save_table(res, "pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia")
cat("\n✓ DE done — res in scope. (GSEA next, once gene sets sorted.)\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VOLCANO — SenHi-DAMhi vs SenHi-DAMlo (microglia)  ·  publication-grade
#   x = logFC (DAMhi − DAMlo)   y = −log10(FDR)
#   right/red = up in DAM+ (DAM/immune)   left/blue = up in DAM− (sen-canonical)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel) })

# ── thresholds ────────────────────────────────────────────────────────────────
FDR_CUT <- 0.05
LFC_CUT <- 0.25                      # modest LFC gate for "called"; tune if you like
N_LABEL <- 14                        # genes to label per side

d <- res %>%
  mutate(neglog10 = -log10(adj.P.Val),
         sig = adj.P.Val < FDR_CUT & abs(logFC) >= LFC_CUT,
         cls = case_when(
           sig & logFC > 0 ~ "DAM+ (immune/effector)",
           sig & logFC < 0 ~ "DAM\u2212 (senescence-canonical)",
           TRUE            ~ "NS"))

# cap y for display (a few genes have astronomically small p) so the cloud is visible
YCAP <- quantile(d$neglog10[is.finite(d$neglog10)], 0.999)
d$neglog10_disp <- pmin(d$neglog10, YCAP)
d$capped <- d$neglog10 > YCAP

# genes to label: top by FDR on each significant side
lab <- bind_rows(
  d %>% filter(cls=="DAM+ (immune/effector)")            %>% arrange(adj.P.Val) %>% head(N_LABEL),
  d %>% filter(cls=="DAM\u2212 (senescence-canonical)")  %>% arrange(adj.P.Val) %>% head(N_LABEL))

PAL <- c("DAM+ (immune/effector)"="#C0392B",
         "DAM\u2212 (senescence-canonical)"="#2471A3",
         "NS"="grey80")

xlim <- max(abs(d$logFC)) * 1.05      # symmetric x

p <- ggplot(d, aes(logFC, neglog10_disp)) +
  # threshold guides
  geom_vline(xintercept=c(-LFC_CUT, LFC_CUT), linetype="dashed", colour="grey70", linewidth=0.3) +
  geom_hline(yintercept=-log10(FDR_CUT), linetype="dashed", colour="grey70", linewidth=0.3) +
  # points: NS first (background), then colored
  geom_point(data=subset(d, cls=="NS"),  aes(colour=cls), size=0.7, alpha=0.35, shape=16) +
  geom_point(data=subset(d, cls!="NS"),  aes(colour=cls), size=1.0, alpha=0.80, shape=16) +
  # capped points get an open triangle at the ceiling
  geom_point(data=subset(d, capped & cls!="NS"), aes(colour=cls),
             y=YCAP, shape=2, size=1.4, stroke=0.5, show.legend=FALSE) +
  geom_text_repel(data=lab, aes(label=gene),
                  colour="black",
                  size=2.5, fontface="italic", segment.size=0.2, segment.color="grey60",
                  min.segment.length=0, box.padding=0.3, max.overlaps=Inf,
                  show.legend=FALSE) +
  scale_colour_manual(values=PAL, name=NULL,
                      breaks=c("DAM\u2212 (senescence-canonical)","DAM+ (immune/effector)")) +
  scale_x_continuous(limits=c(-xlim, xlim)) +
  labs(x=expression(log[2]~fold~change~"(DAM+ vs DAM\u2212, within senescent microglia)"),
       y=expression(-log[10]~FDR),
       title="Senescent microglia: DAM+ vs DAM\u2212 transcriptional programs") +
  theme_classic(base_size=9) +
  theme(plot.title   = element_text(face="bold", size=10, hjust=0),
        axis.title   = element_text(size=9),
        axis.text    = element_text(size=8, colour="black"),
        panel.border = element_rect(colour="black", fill=NA, linewidth=0.5),  # full border
        axis.line    = element_blank(),                                       # drop L/B-only lines
        axis.ticks   = element_line(linewidth=0.4),
        legend.position = c(0.5, 0.97),
        legend.direction = "horizontal",
        legend.text  = element_text(size=8),
        legend.key.size = unit(0.4,"lines"),
        plot.margin  = margin(8,10,6,8))

# count annotations in corners
n_pos <- sum(d$cls=="DAM+ (immune/effector)"); n_neg <- sum(d$cls=="DAM\u2212 (senescence-canonical)")
p <- p +
  annotate("text", x= xlim*0.95, y=0, hjust=1, vjust=0, size=2.6, colour=PAL[1],
           label=sprintf("%d up in DAM+", n_pos)) +
  annotate("text", x=-xlim*0.95, y=0, hjust=0, vjust=0, size=2.6, colour=PAL[2],
           label=sprintf("%d up in DAM\u2212", n_neg))

options(repr.plot.width=5.2, repr.plot.height=4.6)
print(p)
ggsave("volcano_SenHiDAMhi_vs_SenHiDAMlo_microglia.pdf", p, width=5.2, height=4.6, device=cairo_pdf)
ggsave("volcano_SenHiDAMhi_vs_SenHiDAMlo_microglia.png", p, width=5.2, height=4.6, dpi=400, bg="white")
ggsave("volcano_SenHiDAMhi_vs_SenHiDAMlo_microglia.svg", p, width=5.2, height=4.6, dpi=400, bg="white")
cat("\n\u2713 volcano saved\n")

---
## 09 · Panel enrichment of the hits

**Why.** Are the differentially expressed genes enriched for the curated panels, or spread across the transcriptome? Enrichment in a hallmark panel is what connects the gene list back to the senescence claim.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# STEP 1h — panel enrichment of the DE hits (Xhi vs Xlo among SenHi)
# ════════════════════════════════════════════════════════════════════════════
if (!exists("res")) stop("\u2717 run the DE first")

# ═══ CONFIG ═════════════════════════════════════════════════════════════════
Y_TAG <- "Sen"; X_TAG <- "IRM"          # must match the DE that produced `res`
FDR_THRESHOLD <- 0.05
DROP_CIRCULAR <- TRUE                    # remove senescence-defining panels from the test set
CIRCULAR <- c("senescence_score","sen_score","SenMayo","Fridman_Up",
              "SASP","DDR","p53_Targets")   # panels that helped define is_senescent
# ════════════════════════════════════════════════════════════════════════════
XHI <- sprintf("%shi", X_TAG); XLO <- sprintf("%slo", X_TAG)

all_panels <- c(gene_lists, MICROGLIA_STATE_PANELS)
all_panels <- lapply(all_panels, function(g) unique(toupper(as.character(g))))
if (DROP_CIRCULAR) {
    drop <- names(all_panels)[toupper(names(all_panels)) %in% toupper(CIRCULAR)]
    if (length(drop)) cat("dropping circular panels:", paste(drop, collapse=", "), "\n\n")
    all_panels <- all_panels[setdiff(names(all_panels), drop)]
}

universe <- toupper(res$gene)
up   <- toupper(res$gene[res$direction=="Up"])
down <- toupper(res$gene[res$direction=="Down"])
cat(sprintf("Universe %d | Up %d | Down %d\n\n", length(universe), length(up), length(down)))

enrich <- function(hits, panel) {
    pin <- intersect(panel, universe); k <- length(intersect(hits, pin))
    K <- length(pin); n <- length(hits); N <- length(universe)
    data.frame(n_tested=K, deg=k, exp=round(n*K/N,1),
               fold=round((k/n)/(K/N),2),
               p=if(K>0&n>0) phyper(k-1,K,N-K,n,lower.tail=FALSE) else NA)
}

for (lab in c(sprintf("UP (higher in %s)", XHI), sprintf("DOWN (higher in %s)", XLO))) {
    hits <- if (grepl("^UP", lab)) up else down
    cat("=", strrep("=",60), "\n", sep=""); cat(lab, "\n"); cat("=", strrep("=",60), "\n", sep="")
    r <- do.call(rbind, lapply(names(all_panels), function(pn) cbind(panel=pn, enrich(hits, all_panels[[pn]]))))
    r$fdr <- p.adjust(r$p,"BH"); r <- r[order(r$p),]
    cat(sprintf("  %-18s %5s %5s %6s %6s %9s\n","panel","deg","exp","fold","sig",""))
    for (i in seq_len(nrow(r)))
        cat(sprintf("  %-18s %3d/%-3d %5.1f %6.2f  %s  fdr=%.2g\n",
            r$panel[i], r$deg[i], r$n_tested[i], r$exp[i], r$fold[i],
            ifelse(!is.na(r$fdr[i]) & r$fdr[i]<FDR_THRESHOLD,"*","ns"), r$fdr[i]))
    cat("\n")
    assign(sprintf("enrich_%s", if (grepl("^UP", lab)) "up" else "down"), r)
}
cat("\u2713 enrichment done.\n")

---
## 10 · Directional Venns

**Why.** Up- and down-regulated genes handled separately. A gene up on one axis and down on the other is not shared signal, and a direction-blind Venn would count it as overlap.

In [ ]:
suppressPackageStartupMessages({ library(dplyr) })
BASE <- file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")

load_deg <- function(f) {
    r <- read.csv(file.path(BASE, f), stringsAsFactors=FALSE)
    if (!"avg_log2FC" %in% names(r)) r$avg_log2FC <- r$logFC
    if (!"p_val_adj"  %in% names(r)) r$p_val_adj  <- r$adj.P.Val
    if (!"direction"  %in% names(r))
        r$direction <- ifelse(r$p_val_adj<0.05 & r$avg_log2FC>0,"Up",
                       ifelse(r$p_val_adj<0.05 & r$avg_log2FC<0,"Down","NS"))
    r
}
A <- load_deg("pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv")  # senescence axis
B <- load_deg("pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia.csv")  # DAM axis

cat("=", strrep("=",60), "\n", sep="")
cat("DEG CHECK — two contrasts\n")
cat("=", strrep("=",60), "\n", sep="")
for (nm in c("Senescence axis (A)","DAM axis (B)")) {
    r <- if (grepl("^Sen", nm)) A else B
    cat(sprintf("\n%s\n", nm))
    cat(sprintf("  tested: %d | Up: %d | Down: %d | NS: %d\n",
        nrow(r), sum(r$direction=="Up"), sum(r$direction=="Down"), sum(r$direction=="NS")))
    cat(sprintf("  logFC range (sig): [%.2f, %.2f]\n",
        min(r$avg_log2FC[r$direction!="NS"]), max(r$avg_log2FC[r$direction!="NS"])))
}

# significant gene sets
A_up <- A$gene[A$direction=="Up"]; A_dn <- A$gene[A$direction=="Down"]; A_sig <- c(A_up,A_dn)
B_up <- B$gene[B$direction=="Up"]; B_dn <- B$gene[B$direction=="Down"]; B_sig <- c(B_up,B_dn)

cat("\n--- overlap of SIGNIFICANT genes (any direction) ---\n")
cat(sprintf("  A sig: %d | B sig: %d | shared: %d\n",
    length(A_sig), length(B_sig), length(intersect(A_sig,B_sig))))
cat(sprintf("  A-only: %d | B-only: %d\n",
    length(setdiff(A_sig,B_sig)), length(setdiff(B_sig,A_sig))))

cat("\n--- directional overlap ---\n")
cat(sprintf("  Up in both     : %d\n", length(intersect(A_up,B_up))))
cat(sprintf("  Down in both   : %d\n", length(intersect(A_dn,B_dn))))
cat(sprintf("  Up in A, Down in B (opposite): %d\n", length(intersect(A_up,B_dn))))
cat(sprintf("  Down in A, Up in B (opposite): %d\n", length(intersect(A_dn,B_up))))

cat("\n--- shared significant genes (first 30) ---\n")
shared <- intersect(A_sig, B_sig)
print(head(shared, 30))

# tag the tested universe (important caveat for venn: different filterByExpr sets)
cat(sprintf("\n--- tested-gene universes ---\n"))
cat(sprintf("  A tested: %d genes | B tested: %d | shared tested: %d\n",
    nrow(A), nrow(B), length(intersect(A$gene, B$gene))))

assign("A_DE", A, envir=.GlobalEnv); assign("B_DE", B, envir=.GlobalEnv)
cat("\n✓ A_DE, B_DE in scope.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DIRECTIONAL VENNS — Senescence axis vs DAM axis (Up genes | Down genes)
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
if (!exists("A_DE")) stop("✗ run the DEG check first")

A_up <- A_DE$gene[A_DE$direction=="Up"]; A_dn <- A_DE$gene[A_DE$direction=="Down"]
B_up <- B_DE$gene[B_DE$direction=="Up"]; B_dn <- B_DE$gene[B_DE$direction=="Down"]

circle <- function(cx,cy,r=1.1,n=200){t<-seq(0,2*pi,length.out=n);data.frame(x=cx+r*cos(t),y=cy+r*sin(t))}

venn2 <- function(sA, sB, nameA, nameB, ttl, fillA, fillB) {
    both<-length(intersect(sA,sB)); aO<-length(setdiff(sA,sB)); bO<-length(setdiff(sB,sA))
    polys <- rbind(cbind(circle(-0.55,0),grp="A"), cbind(circle(0.55,0),grp="B"))
    ggplot() +
        geom_polygon(data=polys, aes(x,y,group=grp,fill=grp), alpha=0.5, color="grey30", linewidth=0.7) +
        scale_fill_manual(values=c(A=fillA,B=fillB), guide="none") +
        annotate("text", x=-1.1, y=0, label=aO, size=5, fontface="bold") +
        annotate("text", x= 1.1, y=0, label=bO, size=5, fontface="bold") +
        annotate("text", x= 0,   y=0, label=both, size=5.5, fontface="bold") +
        annotate("text", x=-0.95,y=1.35,label=nameA,size=3.6,fontface="bold",color=fillA) +
        annotate("text", x= 0.95,y=1.35,label=nameB,size=3.6,fontface="bold",color=fillB) +
        labs(title=ttl) +
        coord_fixed(xlim=c(-2,2), ylim=c(-1.5,1.8)) +
        theme_void() + theme(plot.title=element_text(size=10,face="bold",hjust=0.5))
}

p_up <- venn2(A_up, B_up, "Sen axis", "DAM axis",
              sprintf("UP genes (Sen %d / DAM %d, %d shared)", length(A_up), length(B_up), length(intersect(A_up,B_up))),
              "#E41A1C", "#B22222")
p_dn <- venn2(A_dn, B_dn, "Sen axis", "DAM axis",
              sprintf("DOWN genes (Sen %d / DAM %d, %d shared)", length(A_dn), length(B_dn), length(intersect(A_dn,B_dn))),
              "#377EB8", "#1F4E79")

panel <- p_up + p_dn +
    plot_annotation(
        title="DEG overlap by direction: Senescence axis vs DAM axis",
        subtitle=sprintf("Note: %d genes are significant in BOTH but OPPOSITE directions (not shown as shared)",
                         length(intersect(A_up,B_dn)) + length(intersect(A_dn,B_up))),
        theme=theme(plot.title=element_text(size=12,face="bold"),
                    plot.subtitle=element_text(size=8,color="#C0392B")))

options(repr.plot.width=10, repr.plot.height=4.5); print(panel)
save_figure(panel, "venn_DEG_directional_Sen_vs_DAM_axis_microglia", width=10, height=4.5)
cat(sprintf("\nUp shared: %d | Down shared: %d | Opposite-direction: %d\n",
    length(intersect(A_up,B_up)), length(intersect(A_dn,B_dn)),
    length(intersect(A_up,B_dn))+length(intersect(A_dn,B_up))))
cat("✓ directional venns done.\n")

In [ ]:
suppressPackageStartupMessages({ library(dplyr) })
if (!exists("A_DE")) stop("✗ run the DEG check first")

A_up <- A_DE$gene[A_DE$direction=="Up"]; A_dn <- A_DE$gene[A_DE$direction=="Down"]
B_up <- B_DE$gene[B_DE$direction=="Up"]; B_dn <- B_DE$gene[B_DE$direction=="Down"]

# helper to print a gene vector wrapped
pgenes <- function(v, perline=8) {
    v <- sort(unique(v))
    if (length(v)==0) { cat("    (none)\n"); return(invisible()) }
    for (i in seq(1, length(v), by=perline))
        cat("    ", paste(v[i:min(i+perline-1, length(v))], collapse=", "), "\n", sep="")
}

cat("=", strrep("=",64), "\n", sep="")
cat("DEG OVERLAP — Senescence axis (A) vs DAM axis (B)\n")
cat("  A = SenHi_DAMlo vs SenLo_DAMlo   B = SenHi_DAMhi vs SenHi_DAMlo\n")
cat("=", strrep("=",64), "\n", sep="")

cat(sprintf("\n■ SHARED UP (up in both) — %d genes:\n", length(intersect(A_up,B_up))))
pgenes(intersect(A_up,B_up))

cat(sprintf("\n■ SHARED DOWN (down in both) — %d genes:\n", length(intersect(A_dn,B_dn))))
pgenes(intersect(A_dn,B_dn))

cat(sprintf("\n■ OPPOSITE — up in Sen axis, DOWN in DAM axis — %d:\n", length(intersect(A_up,B_dn))))
pgenes(intersect(A_up,B_dn))
cat(sprintf("\n■ OPPOSITE — down in Sen axis, UP in DAM axis — %d:\n", length(intersect(A_dn,B_up))))
pgenes(intersect(A_dn,B_up))

cat(sprintf("\n■ UNIQUE to SENESCENCE axis (sig in A, not sig in B) — %d:\n",
            length(setdiff(c(A_up,A_dn), c(B_up,B_dn)))))
A_uniq <- setdiff(c(A_up,A_dn), c(B_up,B_dn))
cat(sprintf("    [up: %d, down: %d]\n",
            length(intersect(A_uniq,A_up)), length(intersect(A_uniq,A_dn))))
pgenes(A_uniq)

cat(sprintf("\n■ UNIQUE to DAM axis (sig in B, not sig in A) — %d:\n",
            length(setdiff(c(B_up,B_dn), c(A_up,A_dn)))))
B_uniq <- setdiff(c(B_up,B_dn), c(A_up,A_dn))
cat(sprintf("    [up: %d, down: %d] — showing top 40 by |logFC|\n",
            length(intersect(B_uniq,B_up)), length(intersect(B_uniq,B_dn))))
# too many to list all — top by |logFC|
b_top <- B_DE[B_DE$gene %in% B_uniq, ]
b_top <- b_top[order(-abs(b_top$avg_log2FC)), ]
pgenes(head(b_top$gene, 40))

# save the categorized lists
cats <- rbind(
    data.frame(gene=intersect(A_up,B_up), category="shared_up"),
    data.frame(gene=intersect(A_dn,B_dn), category="shared_down"),
    data.frame(gene=intersect(A_up,B_dn), category="opposite_AupBdn"),
    data.frame(gene=intersect(A_dn,B_up), category="opposite_AdnBup"),
    data.frame(gene=A_uniq, category="senescence_axis_unique"),
    data.frame(gene=B_uniq, category="DAM_axis_unique")
)
save_table(cats, "DEG_overlap_categories_Sen_vs_DAM_axis")
cat("\n✓ categorized gene lists saved.\n")

---
## 11 · Genome-maintenance genes

**Why.** Focused readout on the gene family carrying the mechanistic claim — log fold change across the senescence axis and the activation axis side by side, so a gene moving on one and not the other reads immediately.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VOLCANO (R house style) — senescence axis, genome-maintenance genes highlighted
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(ggrepel) })
FDR_THRESHOLD <- 0.05

BASE <- file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")
A <- read.csv(file.path(BASE, "pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv"), stringsAsFactors=FALSE)

# genome-maintenance highlight sets (among the senescence-unique genes)
CHROM <- c("AEBP2","DNMT1","KAT2B","KDM1B","KMT2E","RBBP4","SF3B1","SUZ12","TRIM24","NUCKS1","MED13","MED13L","MED4","NIPBL","RBM17")
DDR   <- c("FANCL","NASP","PRIMPOL","PRKDC","RAD54L2","SHLD2","XPA","CENPP","NEK9")
AUTO  <- c("ATG12","WIPI1","NPC1","GGA2","ATP6V1H","PIP4P2","SPPL3")

A$cat <- ifelse(A$gene %in% CHROM, "Chromatin/epigenetic",
         ifelse(A$gene %in% DDR,   "DNA repair",
         ifelse(A$gene %in% AUTO,  "Autophagy/lysosome", "other")))
A$cat <- factor(A$cat, levels=c("Chromatin/epigenetic","DNA repair","Autophagy/lysosome","other"))
A$nlfdr <- -log10(pmax(A$adj.P.Val, 1e-300))

# label highlighted genes that pass FDR<0.10
A$label <- ""
A$label[A$cat!="other" & A$adj.P.Val<0.10] <- A$gene[A$cat!="other" & A$adj.P.Val<0.10]

CATCOL <- c("Chromatin/epigenetic"="#6A3D9A", "DNA repair"="#E41A1C",
            "Autophagy/lysosome"="#FF7F00", "other"="grey75")

p <- ggplot(A, aes(logFC, nlfdr)) +
    # background (non-highlighted): NS pale, sig darker grey
    geom_point(data=subset(A, cat=="other" & adj.P.Val>=FDR_THRESHOLD),
               color="grey85", size=0.8, alpha=0.5) +
    geom_point(data=subset(A, cat=="other" & adj.P.Val<FDR_THRESHOLD),
               color="grey55", size=1.0, alpha=0.6) +
    # highlighted genes on top
    geom_point(data=subset(A, cat!="other"),
               aes(color=cat), size=2.6, alpha=0.9) +
    geom_vline(xintercept=0, linetype="dashed", color="grey40") +
    geom_hline(yintercept=-log10(FDR_THRESHOLD), linetype="dashed", color="grey40") +
    geom_text_repel(aes(label=label, color=cat), size=2.5, max.overlaps=20,
                    fontface="italic", segment.size=0.3, show.legend=FALSE) +
    scale_color_manual(values=CATCOL, name=NULL,
                       breaks=c("Chromatin/epigenetic","DNA repair","Autophagy/lysosome")) +
    labs(title="Senescence axis — genome-maintenance genes",
         subtitle="SenHi_DAMlo vs SenLo_DAMlo",
         x=expression(log[2]~"Fold Change (SenHi - SenLo)"),
         y=expression(-log[10]~"(FDR)")) +
    theme_classic(base_size=10) +
    theme(legend.position="right",
          plot.title=element_text(size=10, face="bold"),
          plot.subtitle=element_text(size=8, color="grey40"),
          panel.border=element_rect(color="black", fill=NA, linewidth=0.5))

options(repr.plot.width=7, repr.plot.height=6); print(p)
save_figure(p, "volcano_senescence_axis_genome_maintenance_microglia", width=7, height=6)
cat("\n✓ R-style volcano done.\n")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HEATMAP — genome-maintenance genes × {senescence axis, DAM axis} logFC
#   no .qs reload — uses the two DE CSVs
# ════════════════════════════════════════════════════════════════════════════
suppressPackageStartupMessages({ library(ggplot2); library(dplyr); library(tidyr) })
BASE <- file.path(SCRATCH, "brain/module_05_senescence_enrichment/disease/AD/psychad_ad/Microglia/results")
A <- read.csv(file.path(BASE,"pseudobulk_DE_SenHiDAMlo_vs_SenLoDAMlo_microglia.csv"), stringsAsFactors=FALSE)  # senescence axis
B <- read.csv(file.path(BASE,"pseudobulk_DE_SenHiDAMhi_vs_SenHiDAMlo_microglia.csv"), stringsAsFactors=FALSE)  # DAM axis

# genome-maintenance gene groups (from the unique-gene functional breakdown)
groups <- list(
 "Chromatin / epigenetic" = c("DNMT1","SUZ12","AEBP2","RBBP4","KAT2B","KDM1B","KMT2E","TRIM24","NIPBL","SF3B1","RBM17","NUCKS1"),
 "DNA repair / DDR"       = c("PRKDC","XPA","FANCL","SHLD2","RAD54L2","PRIMPOL","NASP","CENPP","NEK9"),
 "Autophagy / lysosome"   = c("ATG12","WIPI1","NPC1","GGA2","ATP6V1H","PIP4P2","SPPL3"),
 "Ubiquitin / proteostasis" = c("UBE2N","UBE3C","UBQLN1","UBR3","TRIP12","USP37","SMURF1","RNF145")
)
gene_df <- do.call(rbind, lapply(names(groups), function(g)
    data.frame(gene=groups[[g]], grp=g, stringsAsFactors=FALSE)))

# pull logFC + FDR from each contrast
get_stats <- function(df, genes) df[match(genes, df$gene), c("logFC","adj.P.Val")]
sa <- get_stats(A, gene_df$gene); da <- get_stats(B, gene_df$gene)
hm <- rbind(
    data.frame(gene=gene_df$gene, grp=gene_df$grp, contrast="Senescence axis",
               logFC=sa$logFC, fdr=sa$adj.P.Val),
    data.frame(gene=gene_df$gene, grp=gene_df$grp, contrast="DAM axis",
               logFC=da$logFC, fdr=da$adj.P.Val)
)
hm <- hm[!is.na(hm$logFC), ]   # drop genes not tested in a contrast
hm$contrast <- factor(hm$contrast, levels=c("Senescence axis","DAM axis"))
hm$grp <- factor(hm$grp, levels=names(groups))
# order genes within group by senescence-axis logFC
ord <- A[match(gene_df$gene, A$gene),]; gene_df$saLFC <- ord$logFC
gene_order <- gene_df %>% arrange(grp, desc(saLFC)) %>% pull(gene)
hm$gene <- factor(hm$gene, levels=rev(gene_order))
hm$star <- ifelse(!is.na(hm$fdr) & hm$fdr<0.05, "*", "")

p <- ggplot(hm, aes(contrast, gene, fill=logFC)) +
    geom_tile(color="white", linewidth=0.5) +
    geom_text(aes(label=star), size=4, vjust=0.78, color="black") +
    scale_fill_gradient2(low="#377EB8", mid="white", high="#E41A1C", midpoint=0,
        name=expression(log[2]~"FC"), limits=max(abs(hm$logFC),na.rm=TRUE)*c(-1,1)) +
    facet_grid(grp~., scales="free_y", space="free_y", switch="y") +
    labs(title="Genome-maintenance genes: senescence axis vs DAM axis",
         subtitle="logFC per contrast | * = FDR<0.05 | senescence-axis-unique program",
         x=NULL, y=NULL) +
    theme_minimal(base_size=9) +
    theme(plot.title=element_text(size=11, face="bold"),
          plot.subtitle=element_text(size=7, color="grey45"),
          panel.grid=element_blank(),
          axis.text.x=element_text(size=9, face="bold"),
          axis.text.y=element_text(size=7),
          strip.text.y.left=element_text(size=7.5, face="bold", angle=0),
          strip.placement="outside",
          legend.key.width=unit(0.3,"cm"))

options(repr.plot.width=6, repr.plot.height=9); print(p)
save_figure(p, "heatmap_genome_maintenance_genes_Sen_vs_DAM_axis_microglia", width=6, height=9)
cat("\n✓ genome-maintenance heatmap done.\n")

---
## 12 · Handoff to module 11

The limma tables written by section 04 are the interface. Module 11 reads them
from disk and builds its own rankings, rather than sharing a kernel with this
notebook — so GSEA can be re-run without refitting any model.

**Ranking metric, applied in module 11.** Signed p-value,
`-log10(p) * sign(logFC)`. It orders on evidence rather than effect size, which
keeps low-expression genes with large but noisy fold changes out of the head of
the list.

The axis-depletion check and the functional annotation of axis-unique genes are
Python and also live in module 11, sections 03 and 09.